In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from langdetect import detect
from textblob import TextBlob
from collections import Counter
import time, re, json, pandas as pd, logging
import os
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --------------------------
# CONFIGURATION CORRIGÉE DU DRIVER
# --------------------------
def setup_driver_with_profile():
    """Configure le driver avec un chemin de profil valide"""
    options = Options()
    
    # DÉSACTIVER LE MODE HEADLESS
    # options.add_argument("--headless=new")  # COMMENTEZ CETTE LIGNE
    
    # ⭐⭐ CHEMIN DE PROFIL CORRIGÉ ⭐⭐
    # Utiliser le dossier temporaire ou le dossier utilisateur actuel
    user_profile_dir = os.path.join(os.path.expanduser("~"), "ChromeProfile")
    
    # Créer le dossier s'il n'existe pas
    os.makedirs(user_profile_dir, exist_ok=True)
    
    options.add_argument(f"user-data-dir={user_profile_dir}")
    options.add_argument("--profile-directory=Default")
    
    # Autres options
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=fr")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    # Désactiver les images pour accélérer le chargement
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2
    }
    options.add_experimental_option("prefs", prefs)
    
    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        return driver
    except Exception as e:
        logging.error(f"Erreur création driver: {e}")
        # Fallback: essayer sans profil utilisateur
        options = Options()
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--window-size=1920,1080")
        options.add_argument("--lang=fr")
        return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# --------------------------
# FONCTION DE CONNEXION MANUELLE
# --------------------------
def wait_for_manual_login(driver, timeout=120):
    """
    Attend que l'utilisateur se connecte manuellement à Twitter
    """
    print("=" * 60)
    print("📱 VEUILLEZ VOUS CONNECTER MANUELLEMENT À TWITTER")
    print("⏳ Le script attendra 2 minutes que vous soyez connecté...")
    print("=" * 60)
    
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            current_url = driver.current_url
            if "twitter.com/i/flow/login" in current_url or "x.com/i/flow/login" in current_url:
                print("🔐 Page de connexion détectée - veuillez vous connecter...")
                time.sleep(5)
            elif "twitter.com/home" in current_url or "x.com/home" in current_url:
                print("✅ Connexion réussie !")
                return True
            elif "twitter.com" in current_url or "x.com" in current_url:
                print("✅ Déjà connecté ou page d'accueil chargée")
                return True
            else:
                print(f"🌐 Page actuelle: {current_url}")
                time.sleep(5)
        except Exception as e:
            print(f"⚠️ Erreur lors de la vérification de connexion: {e}")
            time.sleep(5)
    
    print("❌ Timeout - connexion non détectée")
    return False

# --------------------------
# VOS FONCTIONS EXISTANTES
# --------------------------
def detect_langue(texte):
    try:
        lang = detect(texte)
        if lang == "ar": return "Arabe"
        if lang == "fr": return "Français"
        if lang == "en": return "Anglais"
        return "Darija"
    except:
        return "Inconnue"

def detect_polarite(texte):
    try:
        return round(TextBlob(texte).sentiment.polarity, 3)
    except:
        return 0

def detect_categorie(texte):
    t = texte.lower()
    if any(w in t for w in ["sport", "match", "football", "foot"]): return "Sport"
    if any(w in t for w in ["roi", "gouvernement", "politique", "ministre"]): return "Politique"
    if any(w in t for w in ["film", "artiste", "musique"]): return "Art"
    if any(w in t for w in ["économie", "finance", "entreprise"]): return "Économie"
    return "Autre"

def parse_number_from_text(s):
    if not s:
        return 0
    s = s.strip()
    m = re.match(r'([\d\.,]+)\s*([KMkM]?)', s)
    if m:
        num = m.group(1).replace(',', '').replace('.', '')
        letter = m.group(2).upper()
        try:
            base = float(m.group(1).replace(',', ''))
        except:
            base = None
        if letter == 'K':
            return int(base * 1000) if base is not None else 0
        if letter == 'M':
            return int(base * 1_000_000) if base is not None else 0
        try:
            return int(float(m.group(1).replace(',', '')))
        except:
            return 0
    m2 = re.search(r'(\d[\d,\. ]+)', s)
    if m2:
        return int(m2.group(1).replace(',', '').replace(' ', '').split('.')[0])
    return 0

def extract_tweet_data(tweet_element):
    """Extrait les données structurées d'un élément tweet"""
    try:
        tweet_data = {}
        
        # Auteur du tweet
        try:
            author = tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="User-Name"]')
            tweet_data['auteur'] = author.text.split('\n')[0]
            tweet_data['compte'] = author.find_element(By.CSS_SELECTOR, 'a[href*="/"]').get_attribute('href').split('/')[-1]
        except:
            tweet_data['auteur'] = "Non trouvé"
            tweet_data['compte'] = "Non trouvé"
        
        # Contenu du tweet
        try:
            content = tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="tweetText"]')
            tweet_data['contenu'] = content.text
        except:
            tweet_data['contenu'] = "Non trouvé"
        
        # Date et heure
        try:
            time_element = tweet_element.find_element(By.CSS_SELECTOR, 'time')
            tweet_data['date_publication'] = time_element.get_attribute('datetime')
            tweet_data['heure_affichage'] = time_element.text
        except:
            tweet_data['date_publication'] = "Non trouvé"
            tweet_data['heure_affichage'] = "Non trouvé"
        
        # Statistiques d'engagement
        engagement_selectors = {
            'reponses': '[data-testid="reply"]',
            'retweets': '[data-testid="retweet"]', 
            'likes': '[data-testid="like"]'
        }
        
        tweet_data['statistiques'] = {}
        for stat, selector in engagement_selectors.items():
            try:
                element = tweet_element.find_element(By.CSS_SELECTOR, selector)
                spans = element.find_elements(By.TAG_NAME, 'span')
                count = "0"
                for span in spans:
                    text = span.text.strip()
                    if text and text.isdigit():
                        count = text
                        break
                tweet_data['statistiques'][stat] = count
            except:
                tweet_data['statistiques'][stat] = "0"
        
        return tweet_data
        
    except Exception as e:
        logging.error(f"Erreur lors de l'extraction du tweet: {e}")
        return None

def get_video_duration_js(driver):
    try:
        script = """
        let vids = Array.from(document.querySelectorAll('video'));
        let durations = vids.map(v => v.duration || 0);
        return durations;
        """
        durations = driver.execute_script(script)
        if durations:
            for d in durations:
                try:
                    if d and d > 0 and d < 60*60*10:
                        return int(round(d))
                except:
                    continue
    except Exception as e:
        logging.warning(f"get_video_duration_js erreur: {e}")
    return 0

def get_views(driver):
    try:
        candidates = driver.find_elements(By.XPATH, "//*[contains(@aria-label, 'Views') or contains(@aria-label, 'Vues') or contains(@aria-label, 'vue') or contains(@aria-label, 'views')]")
        for el in candidates:
            al = el.get_attribute("aria-label") or ""
            num = parse_number_from_text(al)
            if num > 0:
                logging.info(f"Views trouvé via aria-label: {num}")
                return num
    except Exception as e:
        logging.debug(f"views aria-label err: {e}")

    try:
        for label in ["Views", "Vues", "vues", "views"]:
            els = driver.find_elements(By.XPATH, f"//span[text()='{label}' or contains(text(), '{label}')]")
            for lab in els:
                try:
                    parent = lab.find_element(By.XPATH, "./..")
                    text = parent.text
                    num = parse_number_from_text(text)
                    if num > 0:
                        logging.info(f"Views trouvé via label sibling: {num}")
                        return num
                except:
                    continue
    except Exception as e:
        logging.debug(f"views label err: {e}")

    try:
        page_text = driver.find_element(By.TAG_NAME, "body").text
        m = re.search(r'(\d[\d\.,\s]*\d)\s*(?:views|vues|Views|Vues|views)', page_text)
        if m:
            num = parse_number_from_text(m.group(1))
            if num > 0:
                logging.info(f"Views trouvé via page_text fallback: {num}")
                return num
    except Exception as e:
        logging.debug(f"views fallback err: {e}")

    logging.warning("Nombre de vues non trouvé.")
    return 0

def extract_metrics(driver):
    metrics = {"Replies": 0, "Reposts": 0, "Likes": 0, "Bookmarks": 0, "Views": 0}
    try:
        button_tests = {
            "Likes": "like",
            "Replies": "reply",
            "Bookmarks": "bookmark"
        }
        for metric, testid in button_tests.items():
            try:
                button = driver.find_element(By.XPATH, f"//button[@data-testid='{testid}']")
                aria_label = button.get_attribute("aria-label") or button.text or ""
                num_match = re.search(r'(\d[\d,\.kKmM ]*)', aria_label)
                if num_match:
                    metrics[metric] = parse_number_from_text(num_match.group(1))
                    logging.info(f"{metric} trouvé : {metrics[metric]}")
            except Exception as e:
                logging.debug(f"{metric} non trouvé: {e}")

        for testid in ["retweet_or_repost", "retweet"]:
            try:
                button = driver.find_element(By.XPATH, f"//button[@data-testid='{testid}']")
                aria_label = button.get_attribute("aria-label") or button.text or ""
                num = parse_number_from_text(aria_label)
                if num > 0:
                    metrics["Reposts"] = num
                    break
            except:
                continue

        metrics["Views"] = get_views(driver)
    except Exception as e:
        logging.error(f"Erreur générale métriques : {e}")
    logging.info(f"Métriques extraites: {metrics}")
    return metrics

def get_comments_texts(driver, max_comments=200):
    comments_data = []
    try:
        for _ in range(8):
            driver.execute_script("window.scrollBy(0, 1200);")
            time.sleep(1.2)

        tweet_elements = driver.find_elements(By.CSS_SELECTOR, '[data-testid="tweet"]')
        logging.info(f"Tweets trouvés: {len(tweet_elements)}")
        
        if len(tweet_elements) <= 1:
            logging.warning("Peu de tweets trouvés — peut-être contenu pas chargé totalement.")
        
        for tweet_element in tweet_elements[1:]:
            if len(comments_data) >= max_comments:
                break
            try:
                comment_data = extract_tweet_data(tweet_element)
                if comment_data and comment_data.get('contenu') and comment_data['contenu'] != "Non trouvé":
                    comment_data['langue'] = detect_langue(comment_data['contenu'])
                    comment_data['polarite'] = detect_polarite(comment_data['contenu'])
                    comment_data['categorie'] = detect_categorie(comment_data['contenu'])
                    comments_data.append(comment_data)
            except Exception as e:
                logging.debug(f"Erreur extraction commentaire structuré: {e}")
                continue
                
    except Exception as e:
        logging.error(f"Erreur get_comments_texts: {e}")
    
    logging.info(f"Commentaires récupérés (structurés): {len(comments_data)}")
    return comments_data

def analyze_comments(comments_data):
    analyzed = []
    langs = Counter()
    polarities = Counter()
    word_counter = Counter()
    authors_counter = Counter()

    for comment in comments_data:
        text = comment.get('contenu', '')
        if not text:
            continue
            
        lang = comment.get('langue', detect_langue(text))
        pol_val = comment.get('polarite', detect_polarite(text))
        
        if pol_val > 0.1:
            pol = "positive"
        elif pol_val < -0.1:
            pol = "negative"
        else:
            pol = "neutral"
            
        analyzed.append({
            "text": text, 
            "language": lang, 
            "polarity_value": pol_val, 
            "sentiment": pol,
            "auteur": comment.get('auteur', 'Inconnu'),
            "compte": comment.get('compte', 'Inconnu'),
            "date": comment.get('date_publication', 'Inconnue'),
            "statistiques": comment.get('statistiques', {})
        })
        
        langs[lang] += 1
        polarities[pol] += 1
        authors_counter[comment.get('auteur', 'Inconnu')] += 1
        
        words = re.findall(r"\b\w+\b", text.lower())
        word_counter.update(words)

    total = len(analyzed) or 1
    lang_pct = {k: round(v/total*100, 2) for k,v in langs.items()}
    pol_pct = {k: round(v/total*100, 2) for k,v in polarities.items()}
    top_words = dict(word_counter.most_common(10))
    top_authors = dict(authors_counter.most_common(5))

    stats = {
        "languages": lang_pct, 
        "polarities": pol_pct, 
        "top_words": top_words,
        "top_authors": top_authors,
        "total_comments": total
    }
    return analyzed, stats

# --------------------------
# SCRAPER TWEET AMÉLIORÉ
# --------------------------
def scraper_tweet_ameliore(url, save_csv=True):
    """Version améliorée avec gestion de la connexion"""
    
    # Configuration du driver avec profil
    driver = setup_driver_with_profile()
    if driver is None:
        logging.error("❌ Impossible de créer le driver Chrome")
        return {}

    try:
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        
        logging.info("📥 Ouverture de l'URL Twitter...")
        driver.get("https://twitter.com")
        
        # Attendre la connexion manuelle
        if not wait_for_manual_login(driver):
            logging.error("❌ Échec de la connexion")
            return {}
        
        # Maintenant aller à l'URL spécifique
        logging.info(f"🎯 Navigation vers l'URL cible: {url}")
        driver.get(url)
        
        # Attendre plus longtemps le chargement
        wait = WebDriverWait(driver, 45)
        time.sleep(8)

        # Scroll progressif plus lent
        logging.info("🔄 Chargement du contenu...")
        for i in range(6):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(2)

        # Vérifier si le tweet est chargé
        try:
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="tweet"]')))
            logging.info("✅ Tweet principal chargé")
        except Exception as e:
            logging.warning(f"⚠️ Tweet principal non trouvé immédiatement: {e}")

        # EXTRAIRE LE TWEET PRINCIPAL
        data = {}
        try:
            main_tweet_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="tweet"]')))
            main_tweet_data = extract_tweet_data(main_tweet_element)
            
            if main_tweet_data:
                data["Titre"] = main_tweet_data.get('contenu', '')
                data["Auteur"] = main_tweet_data.get('auteur', '')
                data["Compte"] = main_tweet_data.get('compte', '')
                data["Date publication"] = main_tweet_data.get('date_publication', '')
                logging.info(f"📝 Titre principal: {data['Titre'][:100]}...")
        except Exception as e:
            logging.error(f"❌ Erreur extraction tweet principal: {e}")
            data["Titre"] = ""

        # Continuer avec le reste de votre logique d'extraction...
        data["Catégorie"] = detect_categorie(data.get("Titre", ""))

        # DUREE (JS)
        dur = get_video_duration_js(driver)
        data["Durée"] = f"{dur}s" if dur else "Inconnue"

        # METRICS
        metrics = extract_metrics(driver)
        data["Likes"] = metrics.get("Likes", 0)
        data["Retweets"] = metrics.get("Reposts", 0)
        data["Commentaires"] = metrics.get("Replies", 0)
        data["Nombre de vues"] = metrics.get("Views", 0)
        data["Nombre de partages"] = metrics.get("Reposts", 0)
        data["Nombre de commentaires"] = metrics.get("Replies", 0)

        # COMMENTAIRES
        comments_data = get_comments_texts(driver, max_comments=300)
        analyzed_comments, comment_stats = analyze_comments(comments_data)
        data["comments"] = analyzed_comments
        data["comment_stats"] = comment_stats

        # MOTS PLUS CITÉS
        title_words = re.findall(r"\b\w+\b", data.get("Titre", "").lower())
        overall_counter = Counter(title_words)
        overall_counter.update(comment_stats.get("top_words", {}))
        data["Mots plus cités"] = dict(overall_counter.most_common(10))

        # LANGUE / % LANGUES
        data["Langue"] = detect_langue(data.get("Titre", "")) if data.get("Titre") else "Inconnue"
        data["% Langues"] = comment_stats.get("languages", {data["Langue"]: 100})

        # POLARITE
        data["Polarité"] = detect_polarite(data.get("Titre", ""))
        if data["Polarité"] > 0.1:
            sent = "Positive"
        elif data["Polarité"] < -0.1:
            sent = "Négative"
        else:
            sent = "Neutre"
        data["% Polarité"] = {sent: 100}
        data["% Polarité Twitter"] = comment_stats.get("polarities", {sent: 100})

        data["Lien tweet"] = url

        # Sauvegarde JSON + CSV
        with open("video_hespress_x.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        if save_csv:
            flat = {
                "Titre": data.get("Titre", ""),
                "Auteur": data.get("Auteur", ""),
                "Compte": data.get("Compte", ""),
                "Catégorie": data.get("Catégorie", ""),
                "Date publication": data.get("Date publication", ""),
                "Durée": data.get("Durée", ""),
                "Likes": data.get("Likes", 0),
                "Retweets": data.get("Retweets", 0),
                "Nombre de vues": data.get("Nombre de vues", 0),
                "Nombre de partages": data.get("Nombre de partages", 0),
                "Nombre de commentaires": data.get("Nombre de commentaires", 0),
                "Mots plus cités": json.dumps(data.get("Mots plus cités", {}), ensure_ascii=False),
                "Langue": data.get("Langue", ""),
                "% Langues": json.dumps(data.get("% Langues", {}), ensure_ascii=False),
                "Polarité": data.get("Polarité", 0),
                "% Polarité": json.dumps(data.get("% Polarité", {}), ensure_ascii=False),
                "Lien tweet": data.get("Lien tweet", "")
            }
            df = pd.DataFrame([flat])
            df.to_csv("video_hespress_x.csv", index=False, encoding="utf-8-sig")

        print(json.dumps(data, indent=2, ensure_ascii=False))
        logging.info("💾 Données sauvegardées avec succès!")
        return data

    except Exception as e:
        logging.error(f"❌ Erreur générale du scraper: {e}")
        return {}
    finally:
        # Option: demander avant de fermer
        response = input("Voulez-vous fermer le navigateur? (o/n): ")
        if response.lower() == 'o':
            driver.quit()
            logging.info("🔒 Navigateur fermé")
        else:
            logging.info("🌐 Navigateur laissé ouvert")

# --------------------------
# EXÉCUTION PRINCIPALE
# --------------------------
if __name__ == "__main__":
    url = "https://x.com/hespress/status/1980326222497460307"
    
    print("Choisissez la méthode de connexion:")
    print("1. Connexion manuelle (recommandé)")
    print("2. Mode simple (sans profil)")
    
    choix = input("Votre choix (1 ou 2): ")
    
    if choix == "1":
        scraper_tweet_ameliore(url)
    elif choix == "2":
        # Mode simple sans profil utilisateur
        options = Options()
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--window-size=1920,1080")
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        # ... reste du code
    else:
        print("Choix invalide, utilisation du mode par défaut")
        scraper_tweet_ameliore(url)

Choisissez la méthode de connexion:
1. Connexion manuelle (recommandé)
2. Mode simple (sans profil)


Votre choix (1 ou 2):  1


2025-10-23 21:49:12,461 - INFO - ====== WebDriver manager ======
2025-10-23 21:49:16,405 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-23 21:49:16,529 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-23 21:49:16,655 - INFO - Driver [C:\Users\dell\.wdm\drivers\chromedriver\win64\141.0.7390.122\chromedriver.exe] found in cache
2025-10-23 21:49:18,186 - INFO - 📥 Ouverture de l'URL Twitter...


📱 VEUILLEZ VOUS CONNECTER MANUELLEMENT À TWITTER
⏳ Le script attendra 2 minutes que vous soyez connecté...


2025-10-23 21:49:20,107 - INFO - 🎯 Navigation vers l'URL cible: https://x.com/hespress/status/1980326222497460307


✅ Connexion réussie !


2025-10-23 21:49:29,555 - INFO - 🔄 Chargement du contenu...
2025-10-23 21:49:41,780 - INFO - ✅ Tweet principal chargé
2025-10-23 21:49:42,095 - INFO - 📝 Titre principal: يعز من يشاء         و يذل من يشاء...
2025-10-23 21:49:42,129 - INFO - Likes trouvé : 1
2025-10-23 21:49:42,156 - INFO - Replies trouvé : 0
2025-10-23 21:49:42,246 - INFO - Views trouvé via aria-label: 1
2025-10-23 21:49:42,246 - INFO - Métriques extraites: {'Replies': 0, 'Reposts': 1, 'Likes': 1, 'Bookmarks': 0, 'Views': 1}
2025-10-23 21:49:51,945 - INFO - Tweets trouvés: 5
2025-10-23 21:49:54,016 - INFO - Commentaires récupérés (structurés): 4
2025-10-23 21:49:54,053 - INFO - 💾 Données sauvegardées avec succès!


{
  "Titre": "يعز من يشاء         و يذل من يشاء",
  "Auteur": "Farid MOSBAH",
  "Compte": "FaridM97259",
  "Date publication": "2025-10-21T06:49:50.000Z",
  "Catégorie": "Autre",
  "Durée": "Inconnue",
  "Likes": 1,
  "Retweets": 1,
  "Commentaires": 0,
  "Nombre de vues": 1,
  "Nombre de partages": 1,
  "Nombre de commentaires": 0,
  "comments": [
    {
      "text": "يا ليتهم إحتفلوا ببناء  مساكن لإخوانهم المنكوبين ضحايا زلزال الحوز الذين مازالوا مع معاناتهم.",
      "language": "Arabe",
      "polarity_value": 0.0,
      "sentiment": "neutral",
      "auteur": "Farid MOSBAH",
      "compte": "FaridM97259",
      "date": "2025-10-21T06:47:40.000Z",
      "statistiques": {
        "reponses": "1",
        "retweets": "1",
        "likes": "2"
      }
    },
    {
      "text": "يجب ان نمسح هذه الحدود الموروثة.. حدودنا الحقة ما وراء وادي تافنة مرورا بتلمسان و معسكر و نزولاً حتى عين صالح",
      "language": "Arabe",
      "polarity_value": 0.0,
      "sentiment": "neutral",
      "auteu

Voulez-vous fermer le navigateur? (o/n):  O


2025-10-23 21:50:59,536 - INFO - 🔒 Navigateur fermé


In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from langdetect import detect
from textblob import TextBlob
from collections import Counter
import time, re, json, pandas as pd, logging
import os

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --------------------------
# CONFIGURATION CORRIGÉE DU DRIVER
# --------------------------
def setup_driver_with_profile():
    """Configure le driver avec un chemin de profil valide"""
    options = Options()
    
    # DÉSACTIVER LE MODE HEADLESS
    # options.add_argument("--headless=new")  # COMMENTEZ CETTE LIGNE
    
    # Chemin de profil corrigé
    user_profile_dir = os.path.join(os.path.expanduser("~"), "ChromeProfile")
    os.makedirs(user_profile_dir, exist_ok=True)
    
    options.add_argument(f"user-data-dir={user_profile_dir}")
    options.add_argument("--profile-directory=Default")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=fr")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2
    }
    options.add_experimental_option("prefs", prefs)
    
    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        return driver
    except Exception as e:
        logging.error(f"Erreur création driver: {e}")
        options = Options()
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--window-size=1920,1080")
        options.add_argument("--lang=fr")
        return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# --------------------------
# FONCTION DE CONNEXION MANUELLE
# --------------------------
def wait_for_manual_login(driver, timeout=120):
    """Attend que l'utilisateur se connecte manuellement à Twitter"""
    print("=" * 60)
    print("📱 VEUILLEZ VOUS CONNECTER MANUELLEMENT À TWITTER")
    print("⏳ Le script attendra 2 minutes que vous soyez connecté...")
    print("=" * 60)
    
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            current_url = driver.current_url
            if "twitter.com/i/flow/login" in current_url or "x.com/i/flow/login" in current_url:
                print("🔐 Page de connexion détectée - veuillez vous connecter...")
                time.sleep(5)
            elif "twitter.com/home" in current_url or "x.com/home" in current_url:
                print("✅ Connexion réussie !")
                return True
            elif "twitter.com" in current_url or "x.com" in current_url:
                print("✅ Déjà connecté ou page d'accueil chargée")
                return True
            else:
                print(f"🌐 Page actuelle: {current_url}")
                time.sleep(5)
        except Exception as e:
            print(f"⚠️ Erreur lors de la vérification de connexion: {e}")
            time.sleep(5)
    
    print("❌ Timeout - connexion non détectée")
    return False

# --------------------------
# FONCTIONS UTILITAIRES (conservées)
# --------------------------
def detect_langue(texte):
    try:
        lang = detect(texte)
        if lang == "ar": return "Arabe"
        if lang == "fr": return "Français"
        if lang == "en": return "Anglais"
        return "Darija"
    except:
        return "Inconnue"

def detect_polarite(texte):
    try:
        return round(TextBlob(texte).sentiment.polarity, 3)
    except:
        return 0

def detect_categorie(texte):
    t = texte.lower()
    if any(w in t for w in ["sport", "match", "football", "foot"]): return "Sport"
    if any(w in t for w in ["roi", "gouvernement", "politique", "ministre"]): return "Politique"
    if any(w in t for w in ["film", "artiste", "musique"]): return "Art"
    if any(w in t for w in ["économie", "finance", "entreprise"]): return "Économie"
    return "Autre"

def parse_number_from_text(s):
    if not s:
        return 0
    s = s.strip()
    m = re.match(r'([\d\.,]+)\s*([KMkM]?)', s)
    if m:
        num = m.group(1).replace(',', '').replace('.', '')
        letter = m.group(2).upper()
        try:
            base = float(m.group(1).replace(',', ''))
        except:
            base = None
        if letter == 'K':
            return int(base * 1000) if base is not None else 0
        if letter == 'M':
            return int(base * 1_000_000) if base is not None else 0
        try:
            return int(float(m.group(1).replace(',', '')))
        except:
            return 0
    m2 = re.search(r'(\d[\d,\. ]+)', s)
    if m2:
        return int(m2.group(1).replace(',', '').replace(' ', '').split('.')[0])
    return 0

# --------------------------
# FONCTIONS D'EXTRACTION AMÉLIORÉES
# --------------------------
def extract_main_tweet_data(driver):
    """Extrait spécifiquement les données du tweet principal (Hespress)"""
    try:
        wait = WebDriverWait(driver, 20)
        
        # ⭐⭐ CORRECTION : Attendre et trouver le tweet principal de Hespress ⭐⭐
        # Chercher l'article principal avec l'auteur Hespress
        main_tweet_selector = 'article[data-testid="tweet"]'
        main_tweet_element = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, main_tweet_selector)))
        
        tweet_data = {}
        
        # Auteur du tweet - Vérifier spécifiquement Hespress
        try:
            author_element = main_tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="User-Name"]')
            author_links = author_element.find_elements(By.CSS_SELECTOR, 'a[href*="/"]')
            for link in author_links:
                href = link.get_attribute('href')
                if 'hespress' in href.lower():
                    tweet_data['auteur'] = "Hespress"
                    tweet_data['compte'] = "hespress"
                    break
            else:
                # Si Hespress non trouvé, prendre le premier auteur
                tweet_data['auteur'] = author_element.text.split('\n')[0]
                tweet_data['compte'] = author_links[0].get_attribute('href').split('/')[-1] if author_links else "Non trouvé"
        except:
            tweet_data['auteur'] = "Non trouvé"
            tweet_data['compte'] = "Non trouvé"
        
        # Contenu du tweet principal
        try:
            content = main_tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="tweetText"]')
            tweet_data['contenu'] = content.text
        except:
            tweet_data['contenu'] = "Non trouvé"
        
        # Date et heure
        try:
            time_element = main_tweet_element.find_element(By.CSS_SELECTOR, 'time')
            tweet_data['date_publication'] = time_element.get_attribute('datetime')
            tweet_data['heure_affichage'] = time_element.text
        except:
            tweet_data['date_publication'] = "Non trouvé"
            tweet_data['heure_affichage'] = "Non trouvé"
        
        return tweet_data
        
    except Exception as e:
        logging.error(f"Erreur lors de l'extraction du tweet principal: {e}")
        return None

def get_video_duration_js(driver):
    """Récupère la durée de la vidéo principale"""
    try:
        script = """
        let vids = Array.from(document.querySelectorAll('video'));
        let durations = vids.map(v => v.duration || 0);
        return durations;
        """
        durations = driver.execute_script(script)
        if durations:
            for d in durations:
                try:
                    if d and d > 0 and d < 60*60*10:
                        return int(round(d))
                except:
                    continue
    except Exception as e:
        logging.warning(f"get_video_duration_js erreur: {e}")
    return 0

def get_views(driver):
    """Récupère les vues de la vidéo principale"""
    try:
        # Méthode 1: Chercher dans les aria-label
        candidates = driver.find_elements(By.XPATH, "//*[contains(@aria-label, 'Views') or contains(@aria-label, 'Vues') or contains(@aria-label, 'vue') or contains(@aria-label, 'views')]")
        for el in candidates:
            al = el.get_attribute("aria-label") or ""
            num = parse_number_from_text(al)
            if num > 0:
                logging.info(f"Views trouvé via aria-label: {num}")
                return num
    except Exception as e:
        logging.debug(f"views aria-label err: {e}")

    # Méthode 2: Chercher près des éléments vidéo
    try:
        video_elements = driver.find_elements(By.TAG_NAME, "video")
        for video in video_elements:
            parent = video.find_element(By.XPATH, "./..")
            siblings = parent.find_elements(By.XPATH, ".//*[contains(text(), 'views') or contains(text(), 'vues')]")
            for sibling in siblings:
                text = sibling.text
                num = parse_number_from_text(text)
                if num > 0:
                    logging.info(f"Views trouvé près de la vidéo: {num}")
                    return num
    except Exception as e:
        logging.debug(f"views near video err: {e}")

    logging.warning("Nombre de vues non trouvé.")
    return 0

def extract_main_tweet_metrics(driver):
    """Extrait les métriques du tweet principal"""
    metrics = {"Replies": 0, "Reposts": 0, "Likes": 0, "Views": 0}
    try:
        # Chercher les boutons d'engagement dans le tweet principal
        wait = WebDriverWait(driver, 10)
        main_tweet = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'article[data-testid="tweet"]')))
        
        # Likes
        try:
            like_button = main_tweet.find_element(By.CSS_SELECTOR, '[data-testid="like"]')
            like_text = like_button.text
            metrics["Likes"] = parse_number_from_text(like_text)
        except:
            pass
        
        # Retweets
        try:
            retweet_button = main_tweet.find_element(By.CSS_SELECTOR, '[data-testid="retweet"]')
            retweet_text = retweet_button.text
            metrics["Reposts"] = parse_number_from_text(retweet_text)
        except:
            pass
        
        # Replies
        try:
            reply_button = main_tweet.find_element(By.CSS_SELECTOR, '[data-testid="reply"]')
            reply_text = reply_button.text
            metrics["Replies"] = parse_number_from_text(reply_text)
        except:
            pass
        
        # Views
        metrics["Views"] = get_views(driver)
        
    except Exception as e:
        logging.error(f"Erreur extraction métriques principales: {e}")
    
    logging.info(f"Métriques principales extraites: {metrics}")
    return metrics

def get_comments_texts(driver, max_comments=200):
    """Récupère les commentaires (en excluant le tweet principal)"""
    comments_data = []
    try:
        # Scroll pour charger les commentaires
        for _ in range(8):
            driver.execute_script("window.scrollBy(0, 1200);")
            time.sleep(1.2)

        # Trouver tous les tweets
        tweet_elements = driver.find_elements(By.CSS_SELECTOR, '[data-testid="tweet"]')
        logging.info(f"Tweets trouvés: {len(tweet_elements)}")
        
        if len(tweet_elements) <= 1:
            logging.warning("Peu de tweets trouvés")
        
        # ⭐⭐ CORRECTION : Exclure le tweet principal en vérifiant l'auteur
        for tweet_element in tweet_elements:
            if len(comments_data) >= max_comments:
                break
            try:
                # Vérifier si c'est un commentaire (pas le tweet principal)
                author_info = extract_tweet_author(tweet_element)
                if author_info and "hespress" not in author_info.get('compte', '').lower():
                    comment_data = extract_tweet_data(tweet_element)
                    if comment_data and comment_data.get('contenu') and comment_data['contenu'] != "Non trouvé":
                        comment_data['langue'] = detect_langue(comment_data['contenu'])
                        comment_data['polarite'] = detect_polarite(comment_data['contenu'])
                        comment_data['categorie'] = detect_categorie(comment_data['contenu'])
                        comments_data.append(comment_data)
            except Exception as e:
                logging.debug(f"Erreur extraction commentaire: {e}")
                continue
                
    except Exception as e:
        logging.error(f"Erreur get_comments_texts: {e}")
    
    logging.info(f"Commentaires récupérés: {len(comments_data)}")
    return comments_data

def extract_tweet_author(tweet_element):
    """Extrait uniquement les informations d'auteur d'un tweet"""
    try:
        author_data = {}
        author = tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="User-Name"]')
        author_data['auteur'] = author.text.split('\n')[0]
        author_links = tweet_element.find_elements(By.CSS_SELECTOR, 'a[href*="/"]')
        if author_links:
            author_data['compte'] = author_links[0].get_attribute('href').split('/')[-1]
        return author_data
    except:
        return None

def extract_tweet_data(tweet_element):
    """Extrait les données d'un tweet (pour les commentaires)"""
    try:
        tweet_data = {}
        
        # Auteur
        try:
            author = tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="User-Name"]')
            tweet_data['auteur'] = author.text.split('\n')[0]
            author_links = tweet_element.find_elements(By.CSS_SELECTOR, 'a[href*="/"]')
            if author_links:
                tweet_data['compte'] = author_links[0].get_attribute('href').split('/')[-1]
            else:
                tweet_data['compte'] = "Non trouvé"
        except:
            tweet_data['auteur'] = "Non trouvé"
            tweet_data['compte'] = "Non trouvé"
        
        # Contenu
        try:
            content = tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="tweetText"]')
            tweet_data['contenu'] = content.text
        except:
            tweet_data['contenu'] = "Non trouvé"
        
        # Date
        try:
            time_element = tweet_element.find_element(By.CSS_SELECTOR, 'time')
            tweet_data['date_publication'] = time_element.get_attribute('datetime')
        except:
            tweet_data['date_publication'] = "Non trouvé"
        
        # Statistiques
        engagement_selectors = {
            'reponses': '[data-testid="reply"]',
            'retweets': '[data-testid="retweet"]', 
            'likes': '[data-testid="like"]'
        }
        
        tweet_data['statistiques'] = {}
        for stat, selector in engagement_selectors.items():
            try:
                element = tweet_element.find_element(By.CSS_SELECTOR, selector)
                spans = element.find_elements(By.TAG_NAME, 'span')
                count = "0"
                for span in spans:
                    text = span.text.strip()
                    if text and (text.isdigit() or 'K' in text or 'M' in text):
                        count = text
                        break
                tweet_data['statistiques'][stat] = parse_number_from_text(count)
            except:
                tweet_data['statistiques'][stat] = 0
        
        return tweet_data
        
    except Exception as e:
        logging.error(f"Erreur extraction tweet: {e}")
        return None

def analyze_comments(comments_data):
    """Analyse les commentaires"""
    analyzed = []
    langs = Counter()
    polarities = Counter()
    word_counter = Counter()
    authors_counter = Counter()

    for comment in comments_data:
        text = comment.get('contenu', '')
        if not text:
            continue
            
        lang = comment.get('langue', detect_langue(text))
        pol_val = comment.get('polarite', detect_polarite(text))
        
        if pol_val > 0.1:
            pol = "positive"
        elif pol_val < -0.1:
            pol = "negative"
        else:
            pol = "neutral"
            
        analyzed.append({
            "text": text, 
            "language": lang, 
            "polarity_value": pol_val, 
            "sentiment": pol,
            "auteur": comment.get('auteur', 'Inconnu'),
            "compte": comment.get('compte', 'Inconnu'),
            "date": comment.get('date_publication', 'Inconnue'),
            "statistiques": comment.get('statistiques', {})
        })
        
        langs[lang] += 1
        polarities[pol] += 1
        authors_counter[comment.get('auteur', 'Inconnu')] += 1
        
        words = re.findall(r"\b\w+\b", text.lower())
        word_counter.update(words)

    total = len(analyzed) or 1
    lang_pct = {k: round(v/total*100, 2) for k,v in langs.items()}
    pol_pct = {k: round(v/total*100, 2) for k,v in polarities.items()}
    top_words = dict(word_counter.most_common(10))
    top_authors = dict(authors_counter.most_common(5))

    stats = {
        "languages": lang_pct, 
        "polarities": pol_pct, 
        "top_words": top_words,
        "top_authors": top_authors,
        "total_comments": total
    }
    return analyzed, stats

# --------------------------
# SCRAPER PRINCIPAL CORRIGÉ
# --------------------------
def scraper_tweet_ameliore(url, save_csv=True):
    """Version corrigée qui cible spécifiquement le tweet principal de Hespress"""
    
    driver = setup_driver_with_profile()
    if driver is None:
        logging.error("❌ Impossible de créer le driver Chrome")
        return {}

    try:
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        
        logging.info("📥 Ouverture de l'URL Twitter...")
        driver.get("https://twitter.com")
        
        # Attendre la connexion manuelle
        if not wait_for_manual_login(driver):
            logging.error("❌ Échec de la connexion")
            return {}
        
        # Aller à l'URL spécifique
        logging.info(f"🎯 Navigation vers l'URL cible: {url}")
        driver.get(url)
        
        # Attendre le chargement
        wait = WebDriverWait(driver, 45)
        time.sleep(8)

        # Scroll pour charger le contenu
        logging.info("🔄 Chargement du contenu...")
        for i in range(6):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(2)

        data = {}
        
        # ⭐⭐ CORRECTION : Extraire le tweet principal de Hespress
        logging.info("🔍 Extraction du tweet principal Hespress...")
        main_tweet_data = extract_main_tweet_data(driver)
        
        if main_tweet_data:
            data["Titre"] = main_tweet_data.get('contenu', '')
            data["Auteur"] = main_tweet_data.get('auteur', '')
            data["Compte"] = main_tweet_data.get('compte', '')
            data["Date publication"] = main_tweet_data.get('date_publication', '')
            logging.info(f"📝 Tweet principal trouvé - Auteur: {data['Auteur']}")
        else:
            logging.error("❌ Impossible d'extraire le tweet principal")
            data["Titre"] = ""
            data["Auteur"] = ""
            data["Compte"] = ""
            data["Date publication"] = ""

        # Catégorie
        data["Catégorie"] = detect_categorie(data.get("Titre", ""))

        # Durée vidéo
        dur = get_video_duration_js(driver)
        data["Durée"] = f"{dur}s" if dur else "Inconnue"

        # Métriques du tweet principal
        logging.info("📊 Extraction des métriques du tweet principal...")
        metrics = extract_main_tweet_metrics(driver)
        data["Likes"] = metrics.get("Likes", 0)
        data["Retweets"] = metrics.get("Reposts", 0)
        data["Commentaires"] = metrics.get("Replies", 0)
        data["Nombre de vues"] = metrics.get("Views", 0)
        data["Nombre de partages"] = metrics.get("Reposts", 0)
        data["Nombre de commentaires"] = metrics.get("Replies", 0)

        # Commentaires
        logging.info("💬 Extraction des commentaires...")
        comments_data = get_comments_texts(driver, max_comments=300)
        analyzed_comments, comment_stats = analyze_comments(comments_data)
        data["comments"] = analyzed_comments
        data["comment_stats"] = comment_stats

        # Mots les plus cités
        title_words = re.findall(r"\b\w+\b", data.get("Titre", "").lower())
        overall_counter = Counter(title_words)
        overall_counter.update(comment_stats.get("top_words", {}))
        data["Mots plus cités"] = dict(overall_counter.most_common(10))

        # Langue
        data["Langue"] = detect_langue(data.get("Titre", "")) if data.get("Titre") else "Inconnue"
        data["% Langues"] = comment_stats.get("languages", {data["Langue"]: 100})

        # Polarité
        data["Polarité"] = detect_polarite(data.get("Titre", ""))
        if data["Polarité"] > 0.1:
            sent = "Positive"
        elif data["Polarité"] < -0.1:
            sent = "Négative"
        else:
            sent = "Neutre"
        data["% Polarité"] = {sent: 100}
        data["% Polarité Twitter"] = comment_stats.get("polarities", {sent: 100})

        data["Lien tweet"] = url

        # Sauvegarde
        with open("video_hespress_x.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        if save_csv:
            flat = {
                "Titre": data.get("Titre", ""),
                "Auteur": data.get("Auteur", ""),
                "Compte": data.get("Compte", ""),
                "Catégorie": data.get("Catégorie", ""),
                "Date publication": data.get("Date publication", ""),
                "Durée": data.get("Durée", ""),
                "Likes": data.get("Likes", 0),
                "Retweets": data.get("Retweets", 0),
                "Nombre de vues": data.get("Nombre de vues", 0),
                "Nombre de partages": data.get("Nombre de partages", 0),
                "Nombre de commentaires": data.get("Nombre de commentaires", 0),
                "Mots plus cités": json.dumps(data.get("Mots plus cités", {}), ensure_ascii=False),
                "Langue": data.get("Langue", ""),
                "% Langues": json.dumps(data.get("% Langues", {}), ensure_ascii=False),
                "Polarité": data.get("Polarité", 0),
                "% Polarité": json.dumps(data.get("% Polarité", {}), ensure_ascii=False),
                "Lien tweet": data.get("Lien tweet", "")
            }
            df = pd.DataFrame([flat])
            df.to_csv("video_hespress_x.csv", index=False, encoding="utf-8-sig")

        print(json.dumps(data, indent=2, ensure_ascii=False))
        logging.info("💾 Données sauvegardées avec succès!")
        return data

    except Exception as e:
        logging.error(f"❌ Erreur générale du scraper: {e}")
        return {}
    finally:
        response = input("Voulez-vous fermer le navigateur? (o/n): ")
        if response.lower() == 'o':
            driver.quit()
            logging.info("🔒 Navigateur fermé")
        else:
            logging.info("🌐 Navigateur laissé ouvert")

# --------------------------
# EXÉCUTION PRINCIPALE
# --------------------------
if __name__ == "__main__":
    url = "https://x.com/hespress/status/1980326222497460307"
    
    print("Choisissez la méthode de connexion:")
    print("1. Connexion manuelle (recommandé)")
    print("2. Mode simple (sans profil)")
    
    choix = input("Votre choix (1 ou 2): ")    
    if choix == "1":
        scraper_tweet_ameliore(url)
    elif choix == "2":
        options = Options()
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--window-size=1920,1080")
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    else:
        scraper_tweet_ameliore(url)

Choisissez la méthode de connexion:
1. Connexion manuelle (recommandé)
2. Mode simple (sans profil)


Votre choix (1 ou 2):  1


2025-10-23 22:04:27,041 - INFO - ====== WebDriver manager ======
2025-10-23 22:04:31,395 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-23 22:04:31,514 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-23 22:04:31,628 - INFO - Driver [C:\Users\dell\.wdm\drivers\chromedriver\win64\141.0.7390.122\chromedriver.exe] found in cache
2025-10-23 22:04:33,603 - INFO - 📥 Ouverture de l'URL Twitter...
2025-10-23 22:04:35,247 - INFO - 🎯 Navigation vers l'URL cible: https://x.com/hespress/status/1980326222497460307


📱 VEUILLEZ VOUS CONNECTER MANUELLEMENT À TWITTER
⏳ Le script attendra 2 minutes que vous soyez connecté...
✅ Déjà connecté ou page d'accueil chargée


2025-10-23 22:04:44,656 - INFO - 🔄 Chargement du contenu...
2025-10-23 22:04:56,778 - INFO - 🔍 Extraction du tweet principal Hespress...
2025-10-23 22:04:56,943 - INFO - 📝 Tweet principal trouvé - Auteur: Yassine
2025-10-23 22:04:56,950 - INFO - 📊 Extraction des métriques du tweet principal...
2025-10-23 22:04:57,076 - INFO - Views trouvé via aria-label: 29
2025-10-23 22:04:57,076 - INFO - Métriques principales extraites: {'Replies': 0, 'Reposts': 0, 'Likes': 0, 'Views': 29}
2025-10-23 22:04:57,078 - INFO - 💬 Extraction des commentaires...
2025-10-23 22:05:06,862 - INFO - Tweets trouvés: 5
2025-10-23 22:05:08,350 - INFO - Commentaires récupérés: 5
2025-10-23 22:05:08,376 - INFO - 💾 Données sauvegardées avec succès!


{
  "Titre": "",
  "Auteur": "Yassine",
  "Compte": "YassineYas49235",
  "Date publication": "2025-10-21T04:51:23.000Z",
  "Catégorie": "Autre",
  "Durée": "2s",
  "Likes": 0,
  "Retweets": 0,
  "Commentaires": 0,
  "Nombre de vues": 29,
  "Nombre de partages": 0,
  "Nombre de commentaires": 0,
  "comments": [
    {
      "text": "الكراغلة راه طلقو ليهوم لما مع 1 باش يغسلو طراميهم المكعللة ياجدك",
      "language": "Arabe",
      "polarity_value": 0.0,
      "sentiment": "neutral",
      "auteur": "مارسيل بيجار معذب الكراغلة",
      "compte": "oki35561954",
      "date": "2025-10-20T20:40:18.000Z",
      "statistiques": {
        "reponses": 0,
        "retweets": 0,
        "likes": 1
      }
    },
    {
      "text": "يعز من يشاء         و يذل من يشاء",
      "language": "Arabe",
      "polarity_value": 0.0,
      "sentiment": "neutral",
      "auteur": "Farid MOSBAH",
      "compte": "FaridM97259",
      "date": "2025-10-21T06:49:50.000Z",
      "statistiques": {
        "reponses"

Voulez-vous fermer le navigateur? (o/n):  o


2025-10-23 22:06:13,633 - INFO - 🔒 Navigateur fermé


In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from langdetect import detect
from textblob import TextBlob
from collections import Counter
import time, re, json, pandas as pd, logging
import os

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --------------------------
# CONFIGURATION DU DRIVER
# --------------------------
def setup_driver_with_profile():
    """Configure le driver avec un chemin de profil valide"""
    options = Options()
    
    user_profile_dir = os.path.join(os.path.expanduser("~"), "ChromeProfile")
    os.makedirs(user_profile_dir, exist_ok=True)
    
    options.add_argument(f"user-data-dir={user_profile_dir}")
    options.add_argument("--profile-directory=Default")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=fr")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    prefs = {
        "profile.managed_default_content_settings.images": 1,  # Activer les images pour voir les vidéos
        "profile.default_content_setting_values.notifications": 2
    }
    options.add_experimental_option("prefs", prefs)
    
    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        return driver
    except Exception as e:
        logging.error(f"Erreur création driver: {e}")
        options = Options()
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--window-size=1920,1080")
        options.add_argument("--lang=fr")
        return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# --------------------------
# FONCTION DE CONNEXION MANUELLE
# --------------------------
def wait_for_manual_login(driver, timeout=120):
    """Attend que l'utilisateur se connecte manuellement à Twitter"""
    print("=" * 60)
    print("📱 VEUILLEZ VOUS CONNECTER MANUELLEMENT À TWITTER")
    print("⏳ Le script attendra 2 minutes que vous soyez connecté...")
    print("=" * 60)
    
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            current_url = driver.current_url
            if "twitter.com/i/flow/login" in current_url or "x.com/i/flow/login" in current_url:
                print("🔐 Page de connexion détectée - veuillez vous connecter...")
                time.sleep(5)
            elif "twitter.com/home" in current_url or "x.com/home" in current_url:
                print("✅ Connexion réussie !")
                return True
            elif "twitter.com" in current_url or "x.com" in current_url:
                print("✅ Déjà connecté ou page d'accueil chargée")
                return True
            else:
                print(f"🌐 Page actuelle: {current_url}")
                time.sleep(5)
        except Exception as e:
            print(f"⚠️ Erreur lors de la vérification de connexion: {e}")
            time.sleep(5)
    
    print("❌ Timeout - connexion non détectée")
    return False

# --------------------------
# FONCTIONS UTILITAIRES
# --------------------------
def detect_langue(texte):
    try:
        lang = detect(texte)
        if lang == "ar": return "Arabe"
        if lang == "fr": return "Français"
        if lang == "en": return "Anglais"
        return "Darija"
    except:
        return "Inconnue"

def detect_polarite(texte):
    try:
        return round(TextBlob(texte).sentiment.polarity, 3)
    except:
        return 0

def detect_categorie(texte):
    t = texte.lower()
    if any(w in t for w in ["sport", "match", "football", "foot"]): return "Sport"
    if any(w in t for w in ["roi", "gouvernement", "politique", "ministre"]): return "Politique"
    if any(w in t for w in ["film", "artiste", "musique"]): return "Art"
    if any(w in t for w in ["économie", "finance", "entreprise"]): return "Économie"
    return "Autre"

def parse_number_from_text(s):
    if not s:
        return 0
    s = s.strip()
    m = re.match(r'([\d\.,]+)\s*([KMkM]?)', s)
    if m:
        num = m.group(1).replace(',', '').replace('.', '')
        letter = m.group(2).upper()
        try:
            base = float(m.group(1).replace(',', ''))
        except:
            base = None
        if letter == 'K':
            return int(base * 1000) if base is not None else 0
        if letter == 'M':
            return int(base * 1_000_000) if base is not None else 0
        try:
            return int(float(m.group(1).replace(',', '')))
        except:
            return 0
    m2 = re.search(r'(\d[\d,\. ]+)', s)
    if m2:
        return int(m2.group(1).replace(',', '').replace(' ', '').split('.')[0])
    return 0

# --------------------------
# ⭐ FONCTIONS D'EXTRACTION CORRIGÉES ⭐
# --------------------------
def extract_main_tweet_data(driver):
    """Extrait spécifiquement le PREMIER tweet (tweet principal)"""
    try:
        wait = WebDriverWait(driver, 20)
        
        # Attendre que les tweets soient chargés
        time.sleep(3)
        
        # ⭐ CORRECTION: Prendre le PREMIER article tweet de la page
        all_tweets = driver.find_elements(By.CSS_SELECTOR, 'article[data-testid="tweet"]')
        
        if not all_tweets:
            logging.error("❌ Aucun tweet trouvé sur la page")
            return None
        
        # Le premier tweet est toujours le tweet principal
        main_tweet_element = all_tweets[0]
        logging.info(f"✅ {len(all_tweets)} tweets trouvés, extraction du premier (tweet principal)")
        
        tweet_data = {}
        
        # Auteur du tweet
        try:
            author_element = main_tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="User-Name"]')
            full_text = author_element.text
            lines = full_text.split('\n')
            tweet_data['auteur'] = lines[0] if lines else "Non trouvé"
            
            # Trouver le compte (@username)
            author_links = main_tweet_element.find_elements(By.CSS_SELECTOR, 'a[role="link"]')
            for link in author_links:
                href = link.get_attribute('href') or ""
                if '/status/' not in href and href.count('/') >= 3:
                    username = href.rstrip('/').split('/')[-1]
                    if username and not username.startswith('status'):
                        tweet_data['compte'] = username
                        break
            
            if 'compte' not in tweet_data:
                tweet_data['compte'] = "Non trouvé"
                
            logging.info(f"📝 Auteur trouvé: {tweet_data['auteur']} (@{tweet_data['compte']})")
        except Exception as e:
            logging.error(f"Erreur extraction auteur: {e}")
            tweet_data['auteur'] = "Non trouvé"
            tweet_data['compte'] = "Non trouvé"
        
        # Contenu du tweet principal
        try:
            content = main_tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="tweetText"]')
            tweet_data['contenu'] = content.text
            logging.info(f"📄 Contenu trouvé: {tweet_data['contenu'][:100]}...")
        except Exception as e:
            logging.warning(f"Contenu non trouvé: {e}")
            tweet_data['contenu'] = ""
        
        # Date et heure
        try:
            time_element = main_tweet_element.find_element(By.CSS_SELECTOR, 'time')
            tweet_data['date_publication'] = time_element.get_attribute('datetime')
            tweet_data['heure_affichage'] = time_element.text
            logging.info(f"🕒 Date trouvée: {tweet_data['heure_affichage']}")
        except Exception as e:
            logging.warning(f"Date non trouvée: {e}")
            tweet_data['date_publication'] = "Non trouvé"
            tweet_data['heure_affichage'] = "Non trouvé"
        
        return tweet_data
        
    except Exception as e:
        logging.error(f"❌ Erreur lors de l'extraction du tweet principal: {e}")
        return None

def get_video_duration_js(driver):
    """Récupère la durée de la vidéo principale"""
    try:
        script = """
        let vids = Array.from(document.querySelectorAll('video'));
        if (vids.length > 0) {
            // Prendre la première vidéo (vidéo principale)
            let mainVideo = vids[0];
            return mainVideo.duration || 0;
        }
        return 0;
        """
        duration = driver.execute_script(script)
        if duration and duration > 0 and duration < 60*60*10:
            logging.info(f"🎬 Durée vidéo trouvée: {int(duration)}s")
            return int(round(duration))
    except Exception as e:
        logging.warning(f"Erreur get_video_duration_js: {e}")
    return 0

def get_views(driver):
    """Récupère les vues en cherchant dans plusieurs emplacements"""
    try:
        # Méthode 1: Chercher "Views" dans les span
        view_spans = driver.find_elements(By.XPATH, "//span[contains(text(), 'Views') or contains(text(), 'views') or contains(text(), 'Vues') or contains(text(), 'vues')]")
        for span in view_spans:
            parent = span.find_element(By.XPATH, "./..")
            aria_label = parent.get_attribute("aria-label") or ""
            num = parse_number_from_text(aria_label)
            if num > 0:
                logging.info(f"👁️ Vues trouvées (méthode 1): {num}")
                return num
    except:
        pass

    try:
        # Méthode 2: Chercher dans les aria-label contenant des chiffres
        all_elements = driver.find_elements(By.XPATH, "//*[@aria-label]")
        for el in all_elements:
            aria = el.get_attribute("aria-label") or ""
            if any(keyword in aria.lower() for keyword in ['views', 'vues', 'vue']):
                num = parse_number_from_text(aria)
                if num > 0:
                    logging.info(f"👁️ Vues trouvées (méthode 2): {num}")
                    return num
    except:
        pass

    logging.warning("⚠️ Nombre de vues non trouvé")
    return 0

def extract_main_tweet_metrics(driver):
    """Extrait les métriques du PREMIER tweet (tweet principal)"""
    metrics = {"Replies": 0, "Reposts": 0, "Likes": 0, "Views": 0}
    try:
        time.sleep(2)
        
        # ⭐ Prendre le PREMIER tweet
        all_tweets = driver.find_elements(By.CSS_SELECTOR, 'article[data-testid="tweet"]')
        if not all_tweets:
            logging.error("❌ Aucun tweet pour extraire les métriques")
            return metrics
        
        main_tweet = all_tweets[0]
        logging.info("📊 Extraction des métriques du premier tweet...")
        
        # Replies
        try:
            reply_button = main_tweet.find_element(By.CSS_SELECTOR, '[data-testid="reply"]')
            aria_label = reply_button.get_attribute('aria-label') or ""
            metrics["Replies"] = parse_number_from_text(aria_label)
            logging.info(f"💬 Replies: {metrics['Replies']}")
        except Exception as e:
            logging.debug(f"Replies non trouvés: {e}")
        
        # Retweets
        try:
            retweet_button = main_tweet.find_element(By.CSS_SELECTOR, '[data-testid="retweet"]')
            aria_label = retweet_button.get_attribute('aria-label') or ""
            metrics["Reposts"] = parse_number_from_text(aria_label)
            logging.info(f"🔄 Retweets: {metrics['Reposts']}")
        except Exception as e:
            logging.debug(f"Retweets non trouvés: {e}")
        
        # Likes
        try:
            like_button = main_tweet.find_element(By.CSS_SELECTOR, '[data-testid="like"]')
            aria_label = like_button.get_attribute('aria-label') or ""
            metrics["Likes"] = parse_number_from_text(aria_label)
            logging.info(f"❤️ Likes: {metrics['Likes']}")
        except Exception as e:
            logging.debug(f"Likes non trouvés: {e}")
        
        # Views
        metrics["Views"] = get_views(driver)
        
    except Exception as e:
        logging.error(f"❌ Erreur extraction métriques: {e}")
    
    return metrics

def get_comments_texts(driver, max_comments=200):
    """Récupère les commentaires (en EXCLUANT le premier tweet)"""
    comments_data = []
    try:
        # Scroll pour charger les commentaires
        logging.info("📜 Scroll pour charger les commentaires...")
        for i in range(8):
            driver.execute_script("window.scrollBy(0, 1200);")
            time.sleep(1.5)

        # Trouver tous les tweets
        tweet_elements = driver.find_elements(By.CSS_SELECTOR, '[data-testid="tweet"]')
        logging.info(f"📝 {len(tweet_elements)} tweets trouvés au total")
        
        if len(tweet_elements) <= 1:
            logging.warning("⚠️ Peu de commentaires trouvés")
            return comments_data
        
        # ⭐ CORRECTION: Ignorer le PREMIER tweet (index 0)
        for idx, tweet_element in enumerate(tweet_elements):
            if idx == 0:
                logging.info("⏭️ Premier tweet ignoré (tweet principal)")
                continue
                
            if len(comments_data) >= max_comments:
                break
                
            try:
                comment_data = extract_tweet_data(tweet_element)
                if comment_data and comment_data.get('contenu') and comment_data['contenu'] != "Non trouvé":
                    comment_data['langue'] = detect_langue(comment_data['contenu'])
                    comment_data['polarite'] = detect_polarite(comment_data['contenu'])
                    comment_data['categorie'] = detect_categorie(comment_data['contenu'])
                    comments_data.append(comment_data)
            except Exception as e:
                logging.debug(f"Erreur extraction commentaire {idx}: {e}")
                continue
                
    except Exception as e:
        logging.error(f"❌ Erreur get_comments_texts: {e}")
    
    logging.info(f"✅ {len(comments_data)} commentaires récupérés")
    return comments_data

def extract_tweet_data(tweet_element):
    """Extrait les données d'un tweet (pour les commentaires)"""
    try:
        tweet_data = {}
        
        # Auteur
        try:
            author = tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="User-Name"]')
            tweet_data['auteur'] = author.text.split('\n')[0]
            author_links = tweet_element.find_elements(By.CSS_SELECTOR, 'a[role="link"]')
            for link in author_links:
                href = link.get_attribute('href') or ""
                if '/status/' not in href:
                    username = href.rstrip('/').split('/')[-1]
                    if username:
                        tweet_data['compte'] = username
                        break
            if 'compte' not in tweet_data:
                tweet_data['compte'] = "Non trouvé"
        except:
            tweet_data['auteur'] = "Non trouvé"
            tweet_data['compte'] = "Non trouvé"
        
        # Contenu
        try:
            content = tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="tweetText"]')
            tweet_data['contenu'] = content.text
        except:
            tweet_data['contenu'] = "Non trouvé"
        
        # Date
        try:
            time_element = tweet_element.find_element(By.CSS_SELECTOR, 'time')
            tweet_data['date_publication'] = time_element.get_attribute('datetime')
        except:
            tweet_data['date_publication'] = "Non trouvé"
        
        # Statistiques
        engagement_selectors = {
            'reponses': '[data-testid="reply"]',
            'retweets': '[data-testid="retweet"]', 
            'likes': '[data-testid="like"]'
        }
        
        tweet_data['statistiques'] = {}
        for stat, selector in engagement_selectors.items():
            try:
                element = tweet_element.find_element(By.CSS_SELECTOR, selector)
                aria_label = element.get_attribute('aria-label') or ""
                tweet_data['statistiques'][stat] = parse_number_from_text(aria_label)
            except:
                tweet_data['statistiques'][stat] = 0
        
        return tweet_data
        
    except Exception as e:
        logging.error(f"Erreur extraction tweet: {e}")
        return None

def analyze_comments(comments_data):
    """Analyse les commentaires"""
    analyzed = []
    langs = Counter()
    polarities = Counter()
    word_counter = Counter()
    authors_counter = Counter()

    for comment in comments_data:
        text = comment.get('contenu', '')
        if not text:
            continue
            
        lang = comment.get('langue', detect_langue(text))
        pol_val = comment.get('polarite', detect_polarite(text))
        
        if pol_val > 0.1:
            pol = "positive"
        elif pol_val < -0.1:
            pol = "negative"
        else:
            pol = "neutral"
            
        analyzed.append({
            "text": text, 
            "language": lang, 
            "polarity_value": pol_val, 
            "sentiment": pol,
            "auteur": comment.get('auteur', 'Inconnu'),
            "compte": comment.get('compte', 'Inconnu'),
            "date": comment.get('date_publication', 'Inconnue'),
            "statistiques": comment.get('statistiques', {})
        })
        
        langs[lang] += 1
        polarities[pol] += 1
        authors_counter[comment.get('auteur', 'Inconnu')] += 1
        
        words = re.findall(r"\b\w+\b", text.lower())
        word_counter.update(words)

    total = len(analyzed) or 1
    lang_pct = {k: round(v/total*100, 2) for k,v in langs.items()}
    pol_pct = {k: round(v/total*100, 2) for k,v in polarities.items()}
    top_words = dict(word_counter.most_common(10))
    top_authors = dict(authors_counter.most_common(5))

    stats = {
        "languages": lang_pct, 
        "polarities": pol_pct, 
        "top_words": top_words,
        "top_authors": top_authors,
        "total_comments": total
    }
    return analyzed, stats

# --------------------------
# SCRAPER PRINCIPAL CORRIGÉ
# --------------------------
def scraper_tweet_ameliore(url, save_csv=True):
    """Scraper corrigé qui extrait correctement le tweet principal"""
    
    driver = setup_driver_with_profile()
    if driver is None:
        logging.error("❌ Impossible de créer le driver Chrome")
        return {}

    try:
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        
        logging.info("📥 Ouverture de Twitter/X...")
        driver.get("https://twitter.com")
        
        # Attendre la connexion manuelle
        if not wait_for_manual_login(driver):
            logging.error("❌ Échec de la connexion")
            return {}
        
        # Aller à l'URL spécifique
        logging.info(f"🎯 Navigation vers: {url}")
        driver.get(url)
        
        # Attendre le chargement complet
        time.sleep(10)
        logging.info("⏳ Attente du chargement de la page...")

        # Scroll léger pour s'assurer que tout est chargé
        for i in range(3):
            driver.execute_script("window.scrollBy(0, 500);")
            time.sleep(2)
        
        # Remonter en haut
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(2)

        data = {}
        
        # ⭐ Extraire le PREMIER tweet (tweet principal)
        logging.info("=" * 60)
        logging.info("🔍 EXTRACTION DU TWEET PRINCIPAL (PREMIER TWEET)")
        logging.info("=" * 60)
        main_tweet_data = extract_main_tweet_data(driver)
        
        if main_tweet_data:
            data["Titre"] = main_tweet_data.get('contenu', '')
            data["Auteur"] = main_tweet_data.get('auteur', '')
            data["Compte"] = main_tweet_data.get('compte', '')
            data["Date publication"] = main_tweet_data.get('date_publication', '')
            logging.info(f"✅ Tweet principal extrait - Auteur: {data['Auteur']} (@{data['Compte']})")
        else:
            logging.error("❌ Impossible d'extraire le tweet principal")
            data["Titre"] = ""
            data["Auteur"] = ""
            data["Compte"] = ""
            data["Date publication"] = ""

        # Catégorie
        data["Catégorie"] = detect_categorie(data.get("Titre", ""))

        # Durée vidéo
        dur = get_video_duration_js(driver)
        data["Durée"] = f"{dur}s" if dur else "Inconnue"

        # Métriques du tweet principal
        logging.info("=" * 60)
        logging.info("📊 EXTRACTION DES MÉTRIQUES DU TWEET PRINCIPAL")
        logging.info("=" * 60)
        metrics = extract_main_tweet_metrics(driver)
        data["Likes"] = metrics.get("Likes", 0)
        data["Retweets"] = metrics.get("Reposts", 0)
        data["Commentaires"] = metrics.get("Replies", 0)
        data["Nombre de vues"] = metrics.get("Views", 0)
        data["Nombre de partages"] = metrics.get("Reposts", 0)
        data["Nombre de commentaires"] = metrics.get("Replies", 0)

        # Commentaires
        logging.info("=" * 60)
        logging.info("💬 EXTRACTION DES COMMENTAIRES")
        logging.info("=" * 60)
        comments_data = get_comments_texts(driver, max_comments=300)
        analyzed_comments, comment_stats = analyze_comments(comments_data)
        data["comments"] = analyzed_comments
        data["comment_stats"] = comment_stats

        # Mots les plus cités
        title_words = re.findall(r"\b\w+\b", data.get("Titre", "").lower())
        overall_counter = Counter(title_words)
        overall_counter.update(comment_stats.get("top_words", {}))
        data["Mots plus cités"] = dict(overall_counter.most_common(10))

        # Langue
        data["Langue"] = detect_langue(data.get("Titre", "")) if data.get("Titre") else "Inconnue"
        data["% Langues"] = comment_stats.get("languages", {data["Langue"]: 100})

        # Polarité
        data["Polarité"] = detect_polarite(data.get("Titre", ""))
        if data["Polarité"] > 0.1:
            sent = "Positive"
        elif data["Polarité"] < -0.1:
            sent = "Négative"
        else:
            sent = "Neutre"
        data["% Polarité"] = {sent: 100}
        data["% Polarité Twitter"] = comment_stats.get("polarities", {sent: 100})

        data["Lien tweet"] = url

        # Sauvegarde JSON
        with open("video_hespress_x.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        logging.info("💾 Fichier JSON sauvegardé: video_hespress_x.json")

        # Sauvegarde CSV
        if save_csv:
            flat = {
                "Titre": data.get("Titre", ""),
                "Auteur": data.get("Auteur", ""),
                "Compte": data.get("Compte", ""),
                "Catégorie": data.get("Catégorie", ""),
                "Date publication": data.get("Date publication", ""),
                "Durée": data.get("Durée", ""),
                "Likes": data.get("Likes", 0),
                "Retweets": data.get("Retweets", 0),
                "Nombre de vues": data.get("Nombre de vues", 0),
                "Nombre de partages": data.get("Nombre de partages", 0),
                "Nombre de commentaires": data.get("Nombre de commentaires", 0),
                "Mots plus cités": json.dumps(data.get("Mots plus cités", {}), ensure_ascii=False),
                "Langue": data.get("Langue", ""),
                "% Langues": json.dumps(data.get("% Langues", {}), ensure_ascii=False),
                "Polarité": data.get("Polarité", 0),
                "% Polarité": json.dumps(data.get("% Polarité", {}), ensure_ascii=False),
                "Lien tweet": data.get("Lien tweet", "")
            }
            df = pd.DataFrame([flat])
            df.to_csv("video_hespress_x.csv", index=False, encoding="utf-8-sig")
            logging.info("💾 Fichier CSV sauvegardé: video_hespress_x.csv")

        # Affichage du résumé
        logging.info("=" * 60)
        logging.info("✅ EXTRACTION TERMINÉE - RÉSUMÉ")
        logging.info("=" * 60)
        logging.info(f"📝 Titre: {data.get('Titre', '')[:100]}...")
        logging.info(f"👤 Auteur: {data.get('Auteur')} (@{data.get('Compte')})")
        logging.info(f"❤️ Likes: {data.get('Likes', 0)}")
        logging.info(f"🔄 Retweets: {data.get('Retweets', 0)}")
        logging.info(f"👁️ Vues: {data.get('Nombre de vues', 0)}")
        logging.info(f"💬 Commentaires: {len(comments_data)}")
        logging.info("=" * 60)
        
        return data

    except Exception as e:
        logging.error(f"❌ Erreur générale du scraper: {e}")
        import traceback
        traceback.print_exc()
        return {}
    finally:
        response = input("\n🔒 Voulez-vous fermer le navigateur? (o/n): ")
        if response.lower() == 'o':
            driver.quit()
            logging.info("🔒 Navigateur fermé")
        else:
            logging.info("🌐 Navigateur laissé ouvert pour inspection")

# --------------------------
# EXÉCUTION PRINCIPALE
# --------------------------
if __name__ == "__main__":
    url = "https://x.com/hespress/status/1980326222497460307"
    
    print("=" * 60)
    print("🚀 SCRAPER TWITTER/X - VERSION CORRIGÉE")
    print("=" * 60)
    print("Choisissez la méthode de connexion:")
    print("1. Connexion manuelle (recommandé)")
    print("2. Mode simple (sans profil)")
    
    choix = input("\nVotre choix (1 ou 2): ").strip()
    
    if choix == "1":
        scraper_tweet_ameliore(url)
    elif choix == "2":
        # tu peux appeler la même fonction sans profil si tu veux ajouter une variante plus tard
        scraper_tweet_ameliore(url)
    else:
        print("❌ Choix invalide. Veuillez entrer 1 ou 2.")


🚀 SCRAPER TWITTER/X - VERSION CORRIGÉE
Choisissez la méthode de connexion:
1. Connexion manuelle (recommandé)
2. Mode simple (sans profil)



Votre choix (1 ou 2):  1


2025-10-24 20:08:30,611 - INFO - ====== WebDriver manager ======
2025-10-24 20:08:34,136 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-24 20:08:34,261 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-24 20:08:34,395 - INFO - Driver [C:\Users\dell\.wdm\drivers\chromedriver\win64\141.0.7390.122\chromedriver-win32/chromedriver.exe] found in cache
2025-10-24 20:08:35,876 - INFO - 📥 Ouverture de Twitter/X...
2025-10-24 20:08:37,482 - INFO - 🎯 Navigation vers: https://x.com/hespress/status/1980326222497460307


📱 VEUILLEZ VOUS CONNECTER MANUELLEMENT À TWITTER
⏳ Le script attendra 2 minutes que vous soyez connecté...
✅ Connexion réussie !


2025-10-24 20:08:48,030 - INFO - ⏳ Attente du chargement de la page...
2025-10-24 20:08:56,161 - INFO - ============================================================
2025-10-24 20:08:56,163 - INFO - 🔍 EXTRACTION DU TWEET PRINCIPAL (PREMIER TWEET)
2025-10-24 20:08:56,164 - INFO - ============================================================
2025-10-24 20:08:59,195 - INFO - ✅ 8 tweets trouvés, extraction du premier (tweet principal)
2025-10-24 20:08:59,325 - INFO - 📝 Auteur trouvé: Hespress هسبريس (@hespress)
2025-10-24 20:08:59,422 - INFO - 📄 Contenu trouvé: مغاربة يحتفلون بتتويج الأشبال عند الحدود مع الجزائر

#المغرب #الجزائر #أشبال_الأطلس #كرة_القدم #vira...
2025-10-24 20:08:59,523 - INFO - 🕒 Date trouvée: 7:32 PM · Oct 20, 2025
2025-10-24 20:08:59,524 - INFO - ✅ Tweet principal extrait - Auteur: Hespress هسبريس (@hespress)
2025-10-24 20:08:59,547 - INFO - 🎬 Durée vidéo trouvée: 28s
2025-10-24 20:08:59,549 - INFO - ============================================================
2025-10-24 


🔒 Voulez-vous fermer le navigateur? (o/n):  o


2025-10-24 20:09:31,630 - INFO - 🔒 Navigateur fermé


In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from langdetect import detect
from textblob import TextBlob
from collections import Counter
import time, re, json, pandas as pd, logging
import os

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --------------------------
# CONFIGURATION DU DRIVER
# --------------------------
def setup_driver_with_profile():
    """Configure le driver avec un chemin de profil valide"""
    options = Options()
    
    user_profile_dir = os.path.join(os.path.expanduser("~"), "ChromeProfile")
    os.makedirs(user_profile_dir, exist_ok=True)
    
    options.add_argument(f"user-data-dir={user_profile_dir}")
    options.add_argument("--profile-directory=Default")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=fr")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    prefs = {
        "profile.managed_default_content_settings.images": 1,  # Activer les images pour voir les vidéos
        "profile.default_content_setting_values.notifications": 2
    }
    options.add_experimental_option("prefs", prefs)
    
    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        return driver
    except Exception as e:
        logging.error(f"Erreur création driver: {e}")
        options = Options()
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--window-size=1920,1080")
        options.add_argument("--lang=fr")
        return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# --------------------------
# FONCTION DE CONNEXION MANUELLE
# --------------------------
def wait_for_manual_login(driver, timeout=120):
    """Attend que l'utilisateur se connecte manuellement à Twitter"""
    print("=" * 60)
    print("📱 VEUILLEZ VOUS CONNECTER MANUELLEMENT À TWITTER")
    print("⏳ Le script attendra 2 minutes que vous soyez connecté...")
    print("=" * 60)
    
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            current_url = driver.current_url
            if "twitter.com/i/flow/login" in current_url or "x.com/i/flow/login" in current_url:
                print("🔐 Page de connexion détectée - veuillez vous connecter...")
                time.sleep(5)
            elif "twitter.com/home" in current_url or "x.com/home" in current_url:
                print("✅ Connexion réussie !")
                return True
            elif "twitter.com" in current_url or "x.com" in current_url:
                print("✅ Déjà connecté ou page d'accueil chargée")
                return True
            else:
                print(f"🌐 Page actuelle: {current_url}")
                time.sleep(5)
        except Exception as e:
            print(f"⚠️ Erreur lors de la vérification de connexion: {e}")
            time.sleep(5)
    
    print("❌ Timeout - connexion non détectée")
    return False

# --------------------------
# FONCTIONS UTILITAIRES
# --------------------------
def detect_langue(texte):
    try:
        lang = detect(texte)
        if lang == "ar": return "Arabe"
        if lang == "fr": return "Français"
        if lang == "en": return "Anglais"
        return "Darija"
    except:
        return "Inconnue"

def detect_polarite(texte):
    try:
        return round(TextBlob(texte).sentiment.polarity, 3)
    except:
        return 0

def detect_categorie(texte):
    t = texte.lower()
    if any(w in t for w in ["sport", "match", "football", "foot"]): return "Sport"
    if any(w in t for w in ["roi", "gouvernement", "politique", "ministre"]): return "Politique"
    if any(w in t for w in ["film", "artiste", "musique"]): return "Art"
    if any(w in t for w in ["économie", "finance", "entreprise"]): return "Économie"
    return "Autre"

def parse_number_from_text(s):
    if not s:
        return 0
    s = s.strip()
    m = re.match(r'([\d\.,]+)\s*([KMkM]?)', s)
    if m:
        num = m.group(1).replace(',', '').replace('.', '')
        letter = m.group(2).upper()
        try:
            base = float(m.group(1).replace(',', ''))
        except:
            base = None
        if letter == 'K':
            return int(base * 1000) if base is not None else 0
        if letter == 'M':
            return int(base * 1_000_000) if base is not None else 0
        try:
            return int(float(m.group(1).replace(',', '')))
        except:
            return 0
    m2 = re.search(r'(\d[\d,\. ]+)', s)
    if m2:
        return int(m2.group(1).replace(',', '').replace(' ', '').split('.')[0])
    return 0

# --------------------------
# ⭐ FONCTIONS D'EXTRACTION CORRIGÉES ⭐
# --------------------------
def extract_main_tweet_data(driver):
    """Extrait spécifiquement le PREMIER tweet (tweet principal)"""
    try:
        wait = WebDriverWait(driver, 20)
        
        # Attendre que les tweets soient chargés
        time.sleep(3)
        
        # ⭐ CORRECTION: Prendre le PREMIER article tweet de la page
        all_tweets = driver.find_elements(By.CSS_SELECTOR, 'article[data-testid="tweet"]')
        
        if not all_tweets:
            logging.error("❌ Aucun tweet trouvé sur la page")
            return None
        
        # Le premier tweet est toujours le tweet principal
        main_tweet_element = all_tweets[0]
        logging.info(f"✅ {len(all_tweets)} tweets trouvés, extraction du premier (tweet principal)")
        
        tweet_data = {}
        
        # Auteur du tweet
        try:
            author_element = main_tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="User-Name"]')
            full_text = author_element.text
            lines = full_text.split('\n')
            tweet_data['auteur'] = lines[0] if lines else "Non trouvé"
            
            # Trouver le compte (@username)
            author_links = main_tweet_element.find_elements(By.CSS_SELECTOR, 'a[role="link"]')
            for link in author_links:
                href = link.get_attribute('href') or ""
                if '/status/' not in href and href.count('/') >= 3:
                    username = href.rstrip('/').split('/')[-1]
                    if username and not username.startswith('status'):
                        tweet_data['compte'] = username
                        break
            
            if 'compte' not in tweet_data:
                tweet_data['compte'] = "Non trouvé"
                
            logging.info(f"📝 Auteur trouvé: {tweet_data['auteur']} (@{tweet_data['compte']})")
        except Exception as e:
            logging.error(f"Erreur extraction auteur: {e}")
            tweet_data['auteur'] = "Non trouvé"
            tweet_data['compte'] = "Non trouvé"
        
        # Contenu du tweet principal
        try:
            content = main_tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="tweetText"]')
            tweet_data['contenu'] = content.text
            logging.info(f"📄 Contenu trouvé: {tweet_data['contenu'][:100]}...")
        except Exception as e:
            logging.warning(f"Contenu non trouvé: {e}")
            tweet_data['contenu'] = ""
        
        # Date et heure
        try:
            time_element = main_tweet_element.find_element(By.CSS_SELECTOR, 'time')
            tweet_data['date_publication'] = time_element.get_attribute('datetime')
            tweet_data['heure_affichage'] = time_element.text
            logging.info(f"🕒 Date trouvée: {tweet_data['heure_affichage']}")
        except Exception as e:
            logging.warning(f"Date non trouvée: {e}")
            tweet_data['date_publication'] = "Non trouvé"
            tweet_data['heure_affichage'] = "Non trouvé"
        
        return tweet_data
        
    except Exception as e:
        logging.error(f"❌ Erreur lors de l'extraction du tweet principal: {e}")
        return None

def get_video_duration_js(driver):
    """Récupère la durée de la vidéo principale"""
    try:
        script = """
        let vids = Array.from(document.querySelectorAll('video'));
        if (vids.length > 0) {
            // Prendre la première vidéo (vidéo principale)
            let mainVideo = vids[0];
            return mainVideo.duration || 0;
        }
        return 0;
        """
        duration = driver.execute_script(script)
        if duration and duration > 0 and duration < 60*60*10:
            logging.info(f"🎬 Durée vidéo trouvée: {int(duration)}s")
            return int(round(duration))
    except Exception as e:
        logging.warning(f"Erreur get_video_duration_js: {e}")
    return 0

def get_views(driver):
    """Récupère les vues en cherchant dans plusieurs emplacements"""
    try:
        # Méthode 1: Chercher "Views" dans les span
        view_spans = driver.find_elements(By.XPATH, "//span[contains(text(), 'Views') or contains(text(), 'views') or contains(text(), 'Vues') or contains(text(), 'vues')]")
        for span in view_spans:
            parent = span.find_element(By.XPATH, "./..")
            aria_label = parent.get_attribute("aria-label") or ""
            num = parse_number_from_text(aria_label)
            if num > 0:
                logging.info(f"👁️ Vues trouvées (méthode 1): {num}")
                return num
    except:
        pass

    try:
        # Méthode 2: Chercher dans les aria-label contenant des chiffres
        all_elements = driver.find_elements(By.XPATH, "//*[@aria-label]")
        for el in all_elements:
            aria = el.get_attribute("aria-label") or ""
            if any(keyword in aria.lower() for keyword in ['views', 'vues', 'vue']):
                num = parse_number_from_text(aria)
                if num > 0:
                    logging.info(f"👁️ Vues trouvées (méthode 2): {num}")
                    return num
    except:
        pass

    logging.warning("⚠️ Nombre de vues non trouvé")
    return 0

def extract_main_tweet_metrics(driver):
    """Extrait les métriques du PREMIER tweet (tweet principal)"""
    metrics = {"Replies": 0, "Reposts": 0, "Likes": 0, "Views": 0}
    try:
        time.sleep(2)
        
        # ⭐ Prendre le PREMIER tweet
        all_tweets = driver.find_elements(By.CSS_SELECTOR, 'article[data-testid="tweet"]')
        if not all_tweets:
            logging.error("❌ Aucun tweet pour extraire les métriques")
            return metrics
        
        main_tweet = all_tweets[0]
        logging.info("📊 Extraction des métriques du premier tweet...")
        
        # Replies
        try:
            reply_button = main_tweet.find_element(By.CSS_SELECTOR, '[data-testid="reply"]')
            aria_label = reply_button.get_attribute('aria-label') or ""
            metrics["Replies"] = parse_number_from_text(aria_label)
            logging.info(f"💬 Replies: {metrics['Replies']}")
        except Exception as e:
            logging.debug(f"Replies non trouvés: {e}")
        
        # Retweets
        try:
            retweet_button = main_tweet.find_element(By.CSS_SELECTOR, '[data-testid="retweet"]')
            aria_label = retweet_button.get_attribute('aria-label') or ""
            metrics["Reposts"] = parse_number_from_text(aria_label)
            logging.info(f"🔄 Retweets: {metrics['Reposts']}")
        except Exception as e:
            logging.debug(f"Retweets non trouvés: {e}")
        
        # Likes
        try:
            like_button = main_tweet.find_element(By.CSS_SELECTOR, '[data-testid="like"]')
            aria_label = like_button.get_attribute('aria-label') or ""
            metrics["Likes"] = parse_number_from_text(aria_label)
            logging.info(f"❤️ Likes: {metrics['Likes']}")
        except Exception as e:
            logging.debug(f"Likes non trouvés: {e}")
        
        # Views
        metrics["Views"] = get_views(driver)
        
    except Exception as e:
        logging.error(f"❌ Erreur extraction métriques: {e}")
    
    return metrics

def get_comments_texts(driver, max_comments=200):
    """Récupère les commentaires (en EXCLUANT le premier tweet)"""
    comments_data = []
    try:
        # Scroll pour charger les commentaires
        logging.info("📜 Scroll pour charger les commentaires...")
        for i in range(8):
            driver.execute_script("window.scrollBy(0, 1200);")
            time.sleep(1.5)

        # Trouver tous les tweets
        tweet_elements = driver.find_elements(By.CSS_SELECTOR, '[data-testid="tweet"]')
        logging.info(f"📝 {len(tweet_elements)} tweets trouvés au total")
        
        if len(tweet_elements) <= 1:
            logging.warning("⚠️ Peu de commentaires trouvés")
            return comments_data
        
        # ⭐ CORRECTION: Ignorer le PREMIER tweet (index 0)
        for idx, tweet_element in enumerate(tweet_elements):
            if idx == 0:
                logging.info("⏭️ Premier tweet ignoré (tweet principal)")
                continue
                
            if len(comments_data) >= max_comments:
                break
                
            try:
                comment_data = extract_tweet_data(tweet_element)
                if comment_data and comment_data.get('contenu') and comment_data['contenu'] != "Non trouvé":
                    comment_data['langue'] = detect_langue(comment_data['contenu'])
                    comment_data['polarite'] = detect_polarite(comment_data['contenu'])
                    comment_data['categorie'] = detect_categorie(comment_data['contenu'])
                    comments_data.append(comment_data)
            except Exception as e:
                logging.debug(f"Erreur extraction commentaire {idx}: {e}")
                continue
                
    except Exception as e:
        logging.error(f"❌ Erreur get_comments_texts: {e}")
    
    logging.info(f"✅ {len(comments_data)} commentaires récupérés")
    return comments_data

def extract_tweet_data(tweet_element):
    """Extrait les données d'un tweet (pour les commentaires)"""
    try:
        tweet_data = {}
        
        # Auteur
        try:
            author = tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="User-Name"]')
            tweet_data['auteur'] = author.text.split('\n')[0]
            author_links = tweet_element.find_elements(By.CSS_SELECTOR, 'a[role="link"]')
            for link in author_links:
                href = link.get_attribute('href') or ""
                if '/status/' not in href:
                    username = href.rstrip('/').split('/')[-1]
                    if username:
                        tweet_data['compte'] = username
                        break
            if 'compte' not in tweet_data:
                tweet_data['compte'] = "Non trouvé"
        except:
            tweet_data['auteur'] = "Non trouvé"
            tweet_data['compte'] = "Non trouvé"
        
        # Contenu
        try:
            content = tweet_element.find_element(By.CSS_SELECTOR, '[data-testid="tweetText"]')
            tweet_data['contenu'] = content.text
        except:
            tweet_data['contenu'] = "Non trouvé"
        
        # Date
        try:
            time_element = tweet_element.find_element(By.CSS_SELECTOR, 'time')
            tweet_data['date_publication'] = time_element.get_attribute('datetime')
        except:
            tweet_data['date_publication'] = "Non trouvé"
        
        # Statistiques
        engagement_selectors = {
            'reponses': '[data-testid="reply"]',
            'retweets': '[data-testid="retweet"]', 
            'likes': '[data-testid="like"]'
        }
        
        tweet_data['statistiques'] = {}
        for stat, selector in engagement_selectors.items():
            try:
                element = tweet_element.find_element(By.CSS_SELECTOR, selector)
                aria_label = element.get_attribute('aria-label') or ""
                tweet_data['statistiques'][stat] = parse_number_from_text(aria_label)
            except:
                tweet_data['statistiques'][stat] = 0
        
        return tweet_data
        
    except Exception as e:
        logging.error(f"Erreur extraction tweet: {e}")
        return None

def analyze_comments(comments_data):
    """Analyse les commentaires"""
    analyzed = []
    langs = Counter()
    polarities = Counter()
    word_counter = Counter()
    authors_counter = Counter()

    for comment in comments_data:
        text = comment.get('contenu', '')
        if not text:
            continue
            
        lang = comment.get('langue', detect_langue(text))
        pol_val = comment.get('polarite', detect_polarite(text))
        
        if pol_val > 0.1:
            pol = "positive"
        elif pol_val < -0.1:
            pol = "negative"
        else:
            pol = "neutral"
            
        analyzed.append({
            "text": text, 
            "language": lang, 
            "polarity_value": pol_val, 
            "sentiment": pol,
            "auteur": comment.get('auteur', 'Inconnu'),
            "compte": comment.get('compte', 'Inconnu'),
            "date": comment.get('date_publication', 'Inconnue'),
            "statistiques": comment.get('statistiques', {})
        })
        
        langs[lang] += 1
        polarities[pol] += 1
        authors_counter[comment.get('auteur', 'Inconnu')] += 1
        
        words = re.findall(r"\b\w+\b", text.lower())
        word_counter.update(words)

    total = len(analyzed) or 1
    lang_pct = {k: round(v/total*100, 2) for k,v in langs.items()}
    pol_pct = {k: round(v/total*100, 2) for k,v in polarities.items()}
    top_words = dict(word_counter.most_common(10))
    top_authors = dict(authors_counter.most_common(5))

    stats = {
        "languages": lang_pct, 
        "polarities": pol_pct, 
        "top_words": top_words,
        "top_authors": top_authors,
        "total_comments": total
    }
    return analyzed, stats

# --------------------------
# SCRAPER PRINCIPAL CORRIGÉ
# --------------------------
def scraper_tweet_ameliore(url, save_csv=True):
    """Scraper corrigé qui extrait correctement le tweet principal"""
    
    driver = setup_driver_with_profile()
    if driver is None:
        logging.error("❌ Impossible de créer le driver Chrome")
        return {}

    try:
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        
        logging.info("📥 Ouverture de Twitter/X...")
        driver.get("https://twitter.com")
        
        # Attendre la connexion manuelle
        if not wait_for_manual_login(driver):
            logging.error("❌ Échec de la connexion")
            return {}
        
        # Aller à l'URL spécifique
        logging.info(f"🎯 Navigation vers: {url}")
        driver.get(url)
        
        # Attendre le chargement complet
        time.sleep(10)
        logging.info("⏳ Attente du chargement de la page...")

        # Scroll léger pour s'assurer que tout est chargé
        for i in range(3):
            driver.execute_script("window.scrollBy(0, 500);")
            time.sleep(2)
        
        # Remonter en haut
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(2)

        data = {}
        
        # ⭐ Extraire le PREMIER tweet (tweet principal)
        logging.info("=" * 60)
        logging.info("🔍 EXTRACTION DU TWEET PRINCIPAL (PREMIER TWEET)")
        logging.info("=" * 60)
        main_tweet_data = extract_main_tweet_data(driver)
        
        if main_tweet_data:
            data["Titre"] = main_tweet_data.get('contenu', '')
            data["Auteur"] = main_tweet_data.get('auteur', '')
            data["Compte"] = main_tweet_data.get('compte', '')
            data["Date publication"] = main_tweet_data.get('date_publication', '')
            logging.info(f"✅ Tweet principal extrait - Auteur: {data['Auteur']} (@{data['Compte']})")
        else:
            logging.error("❌ Impossible d'extraire le tweet principal")
            data["Titre"] = ""
            data["Auteur"] = ""
            data["Compte"] = ""
            data["Date publication"] = ""

        # Catégorie
        data["Catégorie"] = detect_categorie(data.get("Titre", ""))

        # Durée vidéo
        dur = get_video_duration_js(driver)
        data["Durée"] = f"{dur}s" if dur else "Inconnue"

        # Métriques du tweet principal
        logging.info("=" * 60)
        logging.info("📊 EXTRACTION DES MÉTRIQUES DU TWEET PRINCIPAL")
        logging.info("=" * 60)
        metrics = extract_main_tweet_metrics(driver)
        data["Likes"] = metrics.get("Likes", 0)
        data["Retweets"] = metrics.get("Reposts", 0)
        data["Commentaires"] = metrics.get("Replies", 0)
        data["Nombre de vues"] = metrics.get("Views", 0)
        data["Nombre de partages"] = metrics.get("Reposts", 0)
        data["Nombre de commentaires"] = metrics.get("Replies", 0)

        # Commentaires
        logging.info("=" * 60)
        logging.info("💬 EXTRACTION DES COMMENTAIRES")
        logging.info("=" * 60)
        comments_data = get_comments_texts(driver, max_comments=300)
        analyzed_comments, comment_stats = analyze_comments(comments_data)
        data["comments"] = analyzed_comments
        data["comment_stats"] = comment_stats

        # Mots les plus cités
        title_words = re.findall(r"\b\w+\b", data.get("Titre", "").lower())
        overall_counter = Counter(title_words)
        overall_counter.update(comment_stats.get("top_words", {}))
        data["Mots plus cités"] = dict(overall_counter.most_common(10))

        # Langue
        data["Langue"] = detect_langue(data.get("Titre", "")) if data.get("Titre") else "Inconnue"
        data["% Langues"] = comment_stats.get("languages", {data["Langue"]: 100})

        # Polarité
        data["Polarité"] = detect_polarite(data.get("Titre", ""))
        if data["Polarité"] > 0.1:
            sent = "Positive"
        elif data["Polarité"] < -0.1:
            sent = "Négative"
        else:
            sent = "Neutre"
        data["% Polarité"] = {sent: 100}
        data["% Polarité Twitter"] = comment_stats.get("polarities", {sent: 100})

        data["Lien tweet"] = url

        # Sauvegarde JSON
        with open("video_hespress_x.json", "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        logging.info("💾 Fichier JSON sauvegardé: video_hespress_x.json")

        # Sauvegarde CSV
        if save_csv:
            flat = {
                "Titre": data.get("Titre", ""),
                "Auteur": data.get("Auteur", ""),
                "Compte": data.get("Compte", ""),
                "Catégorie": data.get("Catégorie", ""),
                "Date publication": data.get("Date publication", ""),
                "Durée": data.get("Durée", ""),
                "Likes": data.get("Likes", 0),
                "Retweets": data.get("Retweets", 0),
                "Nombre de vues": data.get("Nombre de vues", 0),
                "Nombre de partages": data.get("Nombre de partages", 0),
                "Nombre de commentaires": data.get("Nombre de commentaires", 0),
                "Mots plus cités": json.dumps(data.get("Mots plus cités", {}), ensure_ascii=False),
                "Langue": data.get("Langue", ""),
                "% Langues": json.dumps(data.get("% Langues", {}), ensure_ascii=False),
                "Polarité": data.get("Polarité", 0),
                "% Polarité": json.dumps(data.get("% Polarité", {}), ensure_ascii=False),
                "Lien tweet": data.get("Lien tweet", "")
            }
            df = pd.DataFrame([flat])
            df.to_csv("video_hespress_x.csv", index=False, encoding="utf-8-sig")
            logging.info("💾 Fichier CSV sauvegardé: video_hespress_x.csv")

        # Affichage du résumé
        logging.info("=" * 60)
        logging.info("✅ EXTRACTION TERMINÉE - RÉSUMÉ")
        logging.info("=" * 60)
        logging.info(f"📝 Titre: {data.get('Titre', '')[:100]}...")
        logging.info(f"👤 Auteur: {data.get('Auteur')} (@{data.get('Compte')})")
        logging.info(f"❤️ Likes: {data.get('Likes', 0)}")
        logging.info(f"🔄 Retweets: {data.get('Retweets', 0)}")
        logging.info(f"👁️ Vues: {data.get('Nombre de vues', 0)}")
        logging.info(f"💬 Commentaires: {len(comments_data)}")
        logging.info("=" * 60)
        
        return data

    except Exception as e:
        logging.error(f"❌ Erreur générale du scraper: {e}")
        import traceback
        traceback.print_exc()
        return {}
    finally:
        response = input("\n🔒 Voulez-vous fermer le navigateur? (o/n): ")
        if response.lower() == 'o':
            driver.quit()
            logging.info("🔒 Navigateur fermé")
        else:
            logging.info("🌐 Navigateur laissé ouvert pour inspection")

# --------------------------
# EXÉCUTION PRINCIPALE
# --------------------------
if __name__ == "__main__":
    url = "https://x.com/hespress/status/1980326222497460307"
    
    print("=" * 60)
    print("🚀 SCRAPER TWITTER/X - VERSION CORRIGÉE")
    print("=" * 60)
    print("Choisissez la méthode de connexion:")
    print("1. Connexion manuelle (recommandé)")
    print("2. Mode simple (sans profil)")
    
    choix = input("\nVotre choix (1 ou 2): ").strip()
    
    if choix == "1":
        scraper_tweet_ameliore(url)
    elif choix == "2":
        # tu peux appeler la même fonction sans profil si tu veux ajouter une variante plus tard
        scraper_tweet_ameliore(url)
    else:
        print("❌ Choix invalide. Veuillez entrer 1 ou 2.")


🚀 SCRAPER TWITTER/X - VERSION CORRIGÉE
Choisissez la méthode de connexion:
1. Connexion manuelle (recommandé)
2. Mode simple (sans profil)



Votre choix (1 ou 2):  1


2025-10-24 20:15:14,751 - INFO - ====== WebDriver manager ======
2025-10-24 20:15:18,323 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-24 20:15:18,444 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-24 20:15:18,564 - INFO - Driver [C:\Users\dell\.wdm\drivers\chromedriver\win64\141.0.7390.122\chromedriver-win32/chromedriver.exe] found in cache
2025-10-24 20:15:20,185 - INFO - 📥 Ouverture de Twitter/X...
2025-10-24 20:15:21,600 - INFO - 🎯 Navigation vers: https://x.com/hespress/status/1980326222497460307


📱 VEUILLEZ VOUS CONNECTER MANUELLEMENT À TWITTER
⏳ Le script attendra 2 minutes que vous soyez connecté...
✅ Déjà connecté ou page d'accueil chargée


2025-10-24 20:15:33,201 - INFO - ⏳ Attente du chargement de la page...
2025-10-24 20:15:41,729 - INFO - ============================================================
2025-10-24 20:15:41,730 - INFO - 🔍 EXTRACTION DU TWEET PRINCIPAL (PREMIER TWEET)
2025-10-24 20:15:41,733 - INFO - ============================================================
2025-10-24 20:15:44,764 - INFO - ✅ 7 tweets trouvés, extraction du premier (tweet principal)
2025-10-24 20:15:44,847 - INFO - 📝 Auteur trouvé: Hespress هسبريس (@hespress)
2025-10-24 20:15:44,889 - INFO - 📄 Contenu trouvé: مغاربة يحتفلون بتتويج الأشبال عند الحدود مع الجزائر

#المغرب #الجزائر #أشبال_الأطلس #كرة_القدم #vira...
2025-10-24 20:15:44,935 - INFO - 🕒 Date trouvée: 7:32 PM · Oct 20, 2025
2025-10-24 20:15:44,936 - INFO - ✅ Tweet principal extrait - Auteur: Hespress هسبريس (@hespress)
2025-10-24 20:15:44,946 - INFO - 🎬 Durée vidéo trouvée: 28s
2025-10-24 20:15:44,947 - INFO - ============================================================
2025-10-24 


🔒 Voulez-vous fermer le navigateur? (o/n):  o


2025-10-24 20:25:39,763 - INFO - 🔒 Navigateur fermé


In [1]:
# --------------------------
# IMPORTS
# --------------------------
import pandas as pd
import logging
import os
import json
import time
import re
from collections import Counter
from langdetect import detect
from textblob import TextBlob

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

# --------------------------
# CONFIGURATION LOGGING
# --------------------------
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --------------------------
# CONFIGURATION DU DRIVER
# --------------------------
def setup_driver_with_profile():
    options = Options()
    user_profile_dir = os.path.join(os.path.expanduser("~"), "ChromeProfile")
    os.makedirs(user_profile_dir, exist_ok=True)
    options.add_argument(f"user-data-dir={user_profile_dir}")
    options.add_argument("--profile-directory=Default")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=fr")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    prefs = {
        "profile.managed_default_content_settings.images": 1,
        "profile.default_content_setting_values.notifications": 2
    }
    options.add_experimental_option("prefs", prefs)
    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        return driver
    except Exception as e:
        logging.error(f"Erreur création driver: {e}")
        return None

# --------------------------
# FONCTION DE CONNEXION MANUELLE
# --------------------------
def wait_for_manual_login(driver, timeout=120):
    print("=" * 60)
    print("📱 VEUILLEZ VOUS CONNECTER MANUELLEMENT À TWITTER")
    print(f"⏳ Le script attendra {timeout} secondes que vous soyez connecté...")
    print("=" * 60)
    
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            current_url = driver.current_url
            if "twitter.com/i/flow/login" in current_url or "x.com/i/flow/login" in current_url:
                print("🔐 Page de connexion détectée - veuillez vous connecter...")
                time.sleep(5)
            elif "twitter.com/home" in current_url or "x.com/home" in current_url:
                print("✅ Connexion réussie !")
                return True
            else:
                print(f"🌐 Page actuelle: {current_url}")
                time.sleep(5)
        except Exception as e:
            print(f"⚠ Erreur lors de la vérification de connexion: {e}")
            time.sleep(5)
    print("❌ Timeout - connexion non détectée")
    return False

# --------------------------
# FONCTIONS UTILITAIRES
# --------------------------
def detect_langue(texte):
    try:
        lang = detect(texte)
        if lang == "ar": return "Arabe"
        if lang == "fr": return "Français"
        if lang == "en": return "Anglais"
        return "Darija"
    except:
        return "Inconnue"

def detect_polarite(texte):
    try:
        return round(TextBlob(texte).sentiment.polarity, 3)
    except:
        return 0

def detect_categorie(texte):
    t = texte.lower()
    if any(w in t for w in ["sport", "match", "football", "foot"]): return "Sport"
    if any(w in t for w in ["roi", "gouvernement", "politique", "ministre"]): return "Politique"
    if any(w in t for w in ["film", "artiste", "musique"]): return "Art"
    if any(w in t for w in ["économie", "finance", "entreprise"]): return "Économie"
    return "Autre"

def parse_number_from_text(s):
    if not s: return 0
    s = s.strip()
    m = re.match(r'([\d\.,]+)\s*([KMkM]?)', s)
    if m:
        try:
            base = float(m.group(1).replace(',', ''))
        except:
            base = 0
        letter = m.group(2).upper()
        if letter == 'K': return int(base * 1000)
        if letter == 'M': return int(base * 1_000_000)
        return int(base)
    return 0

# --------------------------
# EXTRACTION DU TWEET
# --------------------------
def extract_main_tweet_data(driver):
    try:
        all_tweets = driver.find_elements(By.CSS_SELECTOR, 'article[data-testid="tweet"]')
        if not all_tweets: return None
        main_tweet = all_tweets[0]
        tweet_data = {}
        try:
            author_element = main_tweet.find_element(By.CSS_SELECTOR, '[data-testid="User-Name"]')
            tweet_data['auteur'] = author_element.text.split('\n')[0]
        except:
            tweet_data['auteur'] = "Non trouvé"
        try:
            content = main_tweet.find_element(By.CSS_SELECTOR, '[data-testid="tweetText"]')
            tweet_data['contenu'] = content.text
        except:
            tweet_data['contenu'] = ""
        try:
            time_element = main_tweet.find_element(By.CSS_SELECTOR, 'time')
            tweet_data['date_publication'] = time_element.get_attribute('datetime')
        except:
            tweet_data['date_publication'] = "Non trouvé"
        return tweet_data
    except:
        return None

# --------------------------
# SCRAPER PRINCIPAL
# --------------------------
def scraper_tweet_ameliore(driver, url):
    try:
        driver.get(url)
        time.sleep(5)
        for i in range(3):
            driver.execute_script("window.scrollBy(0, 500);")
            time.sleep(2)
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(2)

        data = extract_main_tweet_data(driver)
        if data:
            data["Catégorie"] = detect_categorie(data.get("contenu", ""))
            data["Lien tweet"] = url
            data["Langue"] = detect_langue(data.get("contenu", ""))
            data["Polarité"] = detect_polarite(data.get("contenu", ""))
        return data
    except Exception as e:
        logging.error(f"❌ Erreur du scraper pour l'URL {url}: {e}")
        return None

# --------------------------
# EXECUTION POUR MULTIPLES URLS
# --------------------------
if __name__ == "__main__":
    fichier_urls = "urls_hesspress_pour_scraping.csv"

    try:
        df_urls = pd.read_csv(fichier_urls)
        urls = df_urls['url_source'].dropna().tolist()
    except Exception as e:
        logging.error(f"❌ Impossible de lire le fichier CSV: {e}")
        urls = []

    if not urls:
        logging.error("❌ Aucune URL à traiter")
        exit()

    driver = setup_driver_with_profile()
    if driver is None or not wait_for_manual_login(driver):
        logging.error("❌ Impossible de lancer le driver ou connexion échouée")
        exit()

    resultats_global = []

    try:
        for idx, url in enumerate(urls, start=1):
            print("="*60)
            print(f"🔹 Scraping URL {idx}/{len(urls)}: {url}")
            print("="*60)
            data = scraper_tweet_ameliore(driver, url)
            if data:
                resultats_global.append(data)
    except KeyboardInterrupt:
        logging.warning("⚠ Scraping stoppé manuellement par l'utilisateur")
    finally:
        driver.quit()

    # Sauvegarde JSON global
    if resultats_global:
        json_file_global = "video_hespress_global.json"
        with open(json_file_global, "w", encoding="utf-8") as f:
            json.dump(resultats_global, f, ensure_ascii=False, indent=2)
        logging.info(f"💾 Fichier JSON global sauvegardé: {json_file_global}")


2025-10-24 21:34:22,800 - INFO - ====== WebDriver manager ======
2025-10-24 21:34:26,314 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-24 21:34:26,432 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-24 21:34:26,548 - INFO - Driver [C:\Users\dell\.wdm\drivers\chromedriver\win64\141.0.7390.122\chromedriver-win32/chromedriver.exe] found in cache


📱 VEUILLEZ VOUS CONNECTER MANUELLEMENT À TWITTER
⏳ Le script attendra 120 secondes que vous soyez connecté...
🌐 Page actuelle: chrome://new-tab-page/
🌐 Page actuelle: chrome://new-tab-page/
🌐 Page actuelle: chrome://new-tab-page/
🌐 Page actuelle: https://x.com/
✅ Connexion réussie !
🔹 Scraping URL 1/157: https://x.com/hespress/status/1977791148388237684
🔹 Scraping URL 2/157: https://x.com/hespress/status/1979940640071758330
🔹 Scraping URL 3/157: https://x.com/hespress/status/1978105974314369213
🔹 Scraping URL 4/157: https://x.com/hespress/status/1978959140849393703
🔹 Scraping URL 5/157: https://x.com/hespress/status/1977464292346667240
🔹 Scraping URL 6/157: https://x.com/hespress/status/1977845774760206777
🔹 Scraping URL 7/157: https://x.com/hespress/status/1979608415857942928
🔹 Scraping URL 8/157: https://x.com/hespress/status/1978204166330736854
🔹 Scraping URL 9/157: https://x.com/hespress/status/1978053669024051312
🔹 Scraping URL 10/157: https://x.com/hespress/status/197672446160576

2025-10-24 21:37:14,881 - ERROR - ❌ Erreur du scraper pour l'URL https://x.com/hespress/status/1976691230336594003: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=141.0.7390.108)
Stacktrace:
	GetHandleVerifier [0x0xb0fe43+66515]
	GetHandleVerifier [0x0xb0fe84+66580]
	(No symbol) [0x0x8fdc48]
	(No symbol) [0x0x8dc18d]
	(No symbol) [0x0x971a4e]
	(No symbol) [0x0x98c4d9]
	(No symbol) [0x0x96afc6]
	(No symbol) [0x0x93c2ca]
	(No symbol) [0x0x93d154]
	GetHandleVerifier [0x0xd67353+2521315]
	GetHandleVerifier [0x0xd622d3+2500707]
	GetHandleVerifier [0x0xb37c94+229924]
	GetHandleVerifier [0x0xb281f8+165768]
	GetHandleVerifier [0x0xb2ecad+193085]
	GetHandleVerifier [0x0xb18158+100072]
	GetHandleVerifier [0x0xb182f0+100480]
	GetHandleVerifier [0x0xb025aa+11066]
	BaseThreadInitThunk [0x0x76155d49+25]
	RtlInitializeExceptionChain [0x0x7738d2fb+107]
	RtlGetAppContainerNamedObjectPath [0x0x7738d281+561]

2025-10-24 21:37:14,885 - 

🔹 Scraping URL 11/157: https://x.com/hespress/status/1976691230336594003
🔹 Scraping URL 12/157: https://x.com/hespress/status/1976586879634968882
🔹 Scraping URL 13/157: https://x.com/hespress/status/1976696776095940652
🔹 Scraping URL 14/157: https://x.com/hespress/status/1976650592513577459
🔹 Scraping URL 15/157: https://x.com/hespress/status/1977685377357078695
🔹 Scraping URL 16/157: https://x.com/hespress/status/1977004431976939625
🔹 Scraping URL 17/157: https://x.com/hespress/status/1978536424010809515
🔹 Scraping URL 18/157: https://x.com/hespress/status/1979932544721174684
🔹 Scraping URL 19/157: https://x.com/hespress/status/1979880297886826916
🔹 Scraping URL 20/157: https://x.com/hespress/status/1978851457525792960
🔹 Scraping URL 21/157: https://x.com/hespress/status/1975939245832610283
🔹 Scraping URL 22/157: https://x.com/hespress/status/1979578223827567015
🔹 Scraping URL 23/157: https://x.com/hespress/status/1976035049234661465
🔹 Scraping URL 24/157: https://x.com/hespress/statu

2025-10-24 21:37:15,066 - ERROR - ❌ Erreur du scraper pour l'URL https://x.com/hespress/status/1978748716065456519: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=141.0.7390.108)
Stacktrace:
	GetHandleVerifier [0x0xb0fe43+66515]
	GetHandleVerifier [0x0xb0fe84+66580]
	(No symbol) [0x0x8fdc48]
	(No symbol) [0x0x8dc18d]
	(No symbol) [0x0x971a4e]
	(No symbol) [0x0x98c4d9]
	(No symbol) [0x0x96afc6]
	(No symbol) [0x0x93c2ca]
	(No symbol) [0x0x93d154]
	GetHandleVerifier [0x0xd67353+2521315]
	GetHandleVerifier [0x0xd622d3+2500707]
	GetHandleVerifier [0x0xb37c94+229924]
	GetHandleVerifier [0x0xb281f8+165768]
	GetHandleVerifier [0x0xb2ecad+193085]
	GetHandleVerifier [0x0xb18158+100072]
	GetHandleVerifier [0x0xb182f0+100480]
	GetHandleVerifier [0x0xb025aa+11066]
	BaseThreadInitThunk [0x0x76155d49+25]
	RtlInitializeExceptionChain [0x0x7738d2fb+107]
	RtlGetAppContainerNamedObjectPath [0x0x7738d281+561]

2025-10-24 21:37:15,070 - 

...

2025-10-24 21:37:15,442 - ERROR - ❌ Erreur du scraper pour l'URL https://x.com/hespress/status/1978415364955373758: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=141.0.7390.108)
Stacktrace:
	GetHandleVerifier [0x0xb0fe43+66515]
	GetHandleVerifier [0x0xb0fe84+66580]
	(No symbol) [0x0x8fdc48]
	(No symbol) [0x0x8dc18d]
	(No symbol) [0x0x971a4e]
	(No symbol) [0x0x98c4d9]
	(No symbol) [0x0x96afc6]
	(No symbol) [0x0x93c2ca]
	(No symbol) [0x0x93d154]
	GetHandleVerifier [0x0xd67353+2521315]
	GetHandleVerifier [0x0xd622d3+2500707]
	GetHandleVerifier [0x0xb37c94+229924]
	GetHandleVerifier [0x0xb281f8+165768]
	GetHandleVerifier [0x0xb2ecad+193085]
	GetHandleVerifier [0x0xb18158+100072]
	GetHandleVerifier [0x0xb182f0+100480]
	GetHandleVerifier [0x0xb025aa+11066]
	BaseThreadInitThunk [0x0x76155d49+25]
	RtlInitializeExceptionChain [0x0x7738d2fb+107]
	RtlGetAppContainerNamedObjectPath [0x0x7738d281+561]

2025-10-24 21:37:15,447 - 

🔹 Scraping URL 111/157: https://x.com/hespress/status/1976978391976100285
🔹 Scraping URL 112/157: https://x.com/hespress/status/1978812937050456163
🔹 Scraping URL 113/157: https://x.com/hespress/status/1976260102853173348
🔹 Scraping URL 114/157: https://x.com/hespress/status/1978513671396483165
🔹 Scraping URL 115/157: https://x.com/hespress/status/1976710748949618942
🔹 Scraping URL 116/157: https://x.com/hespress/status/1976585078340739449
🔹 Scraping URL 117/157: https://x.com/hespress/status/1976033588455112795
🔹 Scraping URL 118/157: https://x.com/hespress/status/1976948239242084740
🔹 Scraping URL 119/157: https://x.com/hespress/status/1977819152849993859
🔹 Scraping URL 120/157: https://x.com/hespress/status/1978506226620105189
🔹 Scraping URL 121/157: https://x.com/hespress/status/1978032466976805046
🔹 Scraping URL 122/157: https://x.com/hespress/status/1975909050660356491
🔹 Scraping URL 123/157: https://x.com/hespress/status/1976754628596289810
🔹 Scraping URL 124/157: https://x.com/

2025-10-24 21:37:15,632 - ERROR - ❌ Erreur du scraper pour l'URL https://x.com/hespress/status/1977702496798732729: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=141.0.7390.108)
Stacktrace:
	GetHandleVerifier [0x0xb0fe43+66515]
	GetHandleVerifier [0x0xb0fe84+66580]
	(No symbol) [0x0x8fdc48]
	(No symbol) [0x0x8dc18d]
	(No symbol) [0x0x971a4e]
	(No symbol) [0x0x98c4d9]
	(No symbol) [0x0x96afc6]
	(No symbol) [0x0x93c2ca]
	(No symbol) [0x0x93d154]
	GetHandleVerifier [0x0xd67353+2521315]
	GetHandleVerifier [0x0xd622d3+2500707]
	GetHandleVerifier [0x0xb37c94+229924]
	GetHandleVerifier [0x0xb281f8+165768]
	GetHandleVerifier [0x0xb2ecad+193085]
	GetHandleVerifier [0x0xb18158+100072]
	GetHandleVerifier [0x0xb182f0+100480]
	GetHandleVerifier [0x0xb025aa+11066]
	BaseThreadInitThunk [0x0x76155d49+25]
	RtlInitializeExceptionChain [0x0x7738d2fb+107]
	RtlGetAppContainerNamedObjectPath [0x0x7738d281+561]

2025-10-24 21:37:15,638 - 

🔹 Scraping URL 139/157: https://x.com/hespress/status/1977702496798732729
🔹 Scraping URL 140/157: https://x.com/hespress/status/1976771524314153397
🔹 Scraping URL 141/157: https://x.com/hespress/status/1978442765106037123
🔹 Scraping URL 142/157: https://x.com/hespress/status/1975969443454079483
🔹 Scraping URL 143/157: https://x.com/hespress/status/1978604313912959119
🔹 Scraping URL 144/157: https://x.com/hespress/status/1978063301503250466
🔹 Scraping URL 145/157: https://x.com/hespress/status/1978046020865986641
🔹 Scraping URL 146/157: https://x.com/hespress/status/1976611174331338850
🔹 Scraping URL 147/157: https://x.com/hespress/status/1976245003392516379
🔹 Scraping URL 148/157: https://x.com/hespress/status/1977705226279825904
🔹 Scraping URL 149/157: https://x.com/hespress/status/1977688034633216343
🔹 Scraping URL 150/157: https://x.com/hespress/status/1979197164757606428
🔹 Scraping URL 151/157: https://x.com/hespress/status/1979276236707738103
🔹 Scraping URL 152/157: https://x.com/

2025-10-24 21:37:18,967 - INFO - 💾 Fichier JSON global sauvegardé: video_hespress_global.json


**code fonctionne pour extraction urls du sep**

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time, re, json, pandas as pd, logging
import os
from urllib.parse import quote, unquote
from datetime import datetime

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --------------------------
# CONFIGURATION DU DRIVER
# --------------------------
def setup_driver():
    """Configure le driver Chrome"""
    options = Options()
    
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=fr")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2
    }
    options.add_experimental_option("prefs", prefs)
    
    try:
        driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
        return driver
    except Exception as e:
        logging.error(f"Erreur création driver: {e}")
        return None

# --------------------------
# FONCTIONS DE SCRAPING GOOGLE AVEC FILTRE TWITTER SEULEMENT
# --------------------------
def search_google_videos_twitter_only(driver, query, max_results=100):
    """Recherche des vidéos sur Google - UNIQUEMENT TWITTER/X"""
    try:
        # Encoder la requête pour l'URL avec filtre site: pour Twitter uniquement
        if "site:x.com" not in query.lower() and "site:twitter.com" not in query.lower():
            query = f"site:x.com {query}"
        
        encoded_query = quote(query)
        url = f"https://www.google.com/search?q={encoded_query}&tbm=vid"
        
        logging.info(f"🔍 Recherche Google Twitter ONLY: {query}")
        driver.get(url)
        time.sleep(4)
        
        # Accepter les cookies si nécessaire
        try:
            accept_button = driver.find_element(By.XPATH, "//button[contains(., 'Tout accepter') or contains(., 'Accept all') or contains(., 'Accepter tout')]")
            accept_button.click()
            time.sleep(2)
        except:
            pass
        
        # Scroll pour charger plus de résultats
        logging.info("📜 Scroll pour charger plus de résultats...")
        for i in range(8):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1.5)
        
        return extract_twitter_links_only(driver, max_results)
        
    except Exception as e:
        logging.error(f"❌ Erreur recherche Google: {e}")
        return []

def extract_twitter_links_only(driver, max_results):
    """Extrait UNIQUEMENT les liens Twitter/X - ignore YouTube, Instagram, etc."""
    twitter_links = []
    
    try:
        # Méthode 1: Chercher spécifiquement les liens Twitter/X
        twitter_selectors = [
            "a[href*='x.com/status/']",
            "a[href*='twitter.com/status/']",
            "a[href*='x.com/']",
            "a[href*='twitter.com/']"
        ]
        
        for selector in twitter_selectors:
            if len(twitter_links) >= max_results:
                break
            try:
                links = driver.find_elements(By.CSS_SELECTOR, selector)
                for link in links:
                    if len(twitter_links) >= max_results:
                        break
                    href = link.get_attribute("href")
                    if href:
                        clean_url = clean_twitter_url(href)
                        if clean_url and clean_url not in twitter_links:
                            twitter_links.append(clean_url)
                            logging.info(f"✅ Lien Twitter trouvé: {clean_url}")
            except:
                continue
        
        # Méthode 2: Filtrer tous les liens pour ne garder que Twitter
        if len(twitter_links) < max_results:
            try:
                all_links = driver.find_elements(By.TAG_NAME, "a")
                for link in all_links:
                    if len(twitter_links) >= max_results:
                        break
                    href = link.get_attribute("href")
                    if href and ("x.com" in href or "twitter.com" in href):
                        clean_url = clean_twitter_url(href)
                        if clean_url and clean_url not in twitter_links:
                            twitter_links.append(clean_url)
                            logging.info(f"✅ Lien Twitter (méthode 2): {clean_url}")
            except:
                pass
        
        # FILTRER les plateformes indésirables
        filtered_links = []
        for link in twitter_links:
            # Exclure explicitement YouTube, Instagram, Facebook, etc.
            if any(platform in link.lower() for platform in ['youtube.com', 'youtu.be', 'instagram.com', 'facebook.com', 'tiktok.com', 'vimeo.com']):
                logging.info(f"🚫 Lien exclu (autre plateforme): {link}")
                continue
            filtered_links.append(link)
        
        logging.info(f"📊 Liens Twitter filtrés: {len(filtered_links)}/{len(twitter_links)}")
        return filtered_links
                
    except Exception as e:
        logging.error(f"❌ Erreur extraction liens Twitter: {e}")
        return []

def clean_twitter_url(url):
    """Nettoie et valide les URLs Twitter - version stricte"""
    try:
        # Si c'est une URL Google redirect, extraire le vrai URL
        if "google.com/url" in url:
            match = re.search(r'url=([^&]+)', url)
            if match:
                url = unquote(match.group(1))
        
        # Garder UNIQUEMENT les URLs Twitter/X avec status
        if ("x.com/" in url or "twitter.com/" in url) and "/status/" in url:
            # Nettoyer l'URL
            clean_url = url.split('?')[0].split('&')[0]
            
            # Vérifier que c'est bien un lien de statut Twitter
            if re.match(r'https?://(x\.com|twitter\.com)/[^/]+/status/\d+', clean_url):
                return clean_url
                
    except Exception as e:
        logging.debug(f"Erreur nettoyage URL: {e}")
    
    return None

def extract_tweet_data_quick(driver):
    """Extrait rapidement les données du tweet (optimisé pour la vitesse)"""
    tweet_data = {
        "title": "",
        "timestamp": "",
        "author": "Hespress",
        "metrics": {"likes": 0, "retweets": 0}
    }
    
    try:
        # Titre du tweet - méthode rapide
        try:
            title_elements = driver.find_elements(By.CSS_SELECTOR, '[data-testid="tweetText"], article div[dir="auto"]')
            if title_elements:
                tweet_data["title"] = title_elements[0].text[:500]  # Limiter la longueur
        except:
            pass
        
        # Date - méthode rapide
        try:
            time_elements = driver.find_elements(By.TAG_NAME, "time")
            if time_elements:
                tweet_data["timestamp"] = time_elements[0].get_attribute("datetime")
        except:
            pass
            
    except Exception as e:
        logging.debug(f"Erreur extraction rapide données tweet: {e}")
    
    return tweet_data

# --------------------------
# SCRAPER PRINCIPAL TWITTER SEULEMENT
# --------------------------
def scrape_hespress_twitter_september_2025():
    """Scrape UNIQUEMENT les vidéos Twitter/X Hespress de septembre 2025"""
    
    driver = setup_driver()
    if not driver:
        logging.error("❌ Impossible de créer le driver Chrome")
        return
    
    try:
        # REQUÊTES SPÉCIFIQUES POUR TWITTER SEULEMENT
        queries = [
            "site:x.com hespress video september 2025",
            "site:x.com hespress septembre 2025",
            "hespress video x.com september 2025",
            "hespress x.com status september 2025",
            "hespress twitter.com video september 2025",
            "site:twitter.com hespress video 2025",
            '"hespress" "x.com" "september 2025"',
            '"hespress" "twitter.com" "septembre 2025"',
            'hespress "2025-09" site:x.com',
            'hespress.com video x.com september',
            # Requêtes plus larges mais filtrées
            "hespress video september 2025 x.com",
            "hespress septembre 2025 twitter.com"
        ]
        
        all_twitter_links = []
        
        for i, query in enumerate(queries, 1):
            logging.info(f"🎯 Recherche Twitter {i}/{len(queries)}: {query}")
            links = search_google_videos_twitter_only(driver, query, max_results=100)
            all_twitter_links.extend(links)
            
            # Sauvegarde intermédiaire
            unique_so_far = list(set(all_twitter_links))
            with open("twitter_urls_temporaire.txt", "w", encoding="utf-8") as f:
                for url in unique_so_far:
                    f.write(url + "\n")
            
            logging.info(f"📊 Progression: {len(unique_so_far)} liens Twitter accumulés")
            time.sleep(2)
        
        # Supprimer les doublons
        unique_links = list(set(all_twitter_links))
        logging.info(f"🎉 TOTAL LIENS TWITTER UNIQUES: {len(unique_links)}")
        
        if not unique_links:
            logging.warning("❌ Aucun lien Twitter trouvé pour septembre 2025.")
            return
        
        # Sauvegarder la liste complète des URLs Twitter
        with open("hespress_twitter_urls_septembre_2025_COMPLET.txt", "w", encoding="utf-8") as f:
            for url in unique_links:
                f.write(url + "\n")
        
        # ANALYSE ET FILTRAGE SEPTEMBRE 2025
        september_2025_tweets = []
        total_a_analyser = len(unique_links)
        
        logging.info(f"🔍 Analyse des {total_a_analyser} tweets pour septembre 2025...")
        
        for i, link in enumerate(unique_links, 1):
            try:
                if i % 10 == 0:
                    logging.info(f"📊 Analyse {i}/{total_a_analyser} - {len(september_2025_tweets)} tweets septembre 2025 validés")
                
                driver.get(link)
                time.sleep(2)
                
                # Extraction rapide des données
                tweet_data = extract_tweet_data_quick(driver)
                
                # FILTRE SEPTEMBRE 2025
                timestamp = tweet_data.get("timestamp", "")
                if "2025-09" in timestamp:
                    tweet_info = {
                        "tweet_url": link,
                        "title": tweet_data["title"],
                        "timestamp": timestamp,
                        "author": tweet_data["author"],
                        "metrics": tweet_data["metrics"],
                        "success": True,
                        "scraped_at": datetime.now().isoformat()
                    }
                    september_2025_tweets.append(tweet_info)
                    logging.info(f"✅ SEPTEMBRE 2025: {timestamp} - {tweet_data['title'][:100]}...")
                else:
                    logging.info(f"⏭️ Tweet ignoré (pas septembre 2025): {timestamp}")
                
                # Sauvegarde incrémentale
                if i % 20 == 0:
                    with open("hespress_twitter_septembre_2025_temp.json", "w", encoding="utf-8") as f:
                        json.dump(september_2025_tweets, f, ensure_ascii=False, indent=2)
                
            except Exception as e:
                logging.debug(f"❌ Erreur sur {link}: {e}")
                continue
        
        # SAUVEGARDE FINALE
        save_twitter_results(september_2025_tweets, len(unique_links))
        
        # STATISTIQUES
        logging.info(f"📊 RAPPORT FINAL TWITTER SEPTEMBRE 2025:")
        logging.info(f"   🔗 Total tweets analysés: {len(unique_links)}")
        logging.info(f"   ✅ Tweets septembre 2025 validés: {len(september_2025_tweets)}")
        logging.info(f"   📈 Taux de réussite: {len(september_2025_tweets)/len(unique_links)*100:.1f}%" if unique_links else "0%")
        
        if september_2025_tweets:
            dates = [result["timestamp"] for result in september_2025_tweets if result.get("timestamp")]
            if dates:
                logging.info(f"   📅 Plage de dates: {min(dates)} à {max(dates)}")
        
    except Exception as e:
        logging.error(f"❌ Erreur générale: {e}")
    finally:
        driver.quit()
        logging.info("🔒 Navigateur fermé")

def save_twitter_results(results, total_analyses):
    """Sauvegarde les résultats Twitter uniquement"""
    try:
        # JSON complet
        output_data = {
            "metadata": {
                "scraping_date": datetime.now().isoformat(),
                "total_twitter_urls_analyzed": total_analyses,
                "september_2025_tweets_found": len(results),
                "success_rate": f"{(len(results)/total_analyses*100):.1f}%" if total_analyses > 0 else "0%",
                "platform": "Twitter/X uniquement"
            },
            "tweets": results
        }
        
        with open("hespress_TWITTER_septembre_2025_FINAL.json", "w", encoding="utf-8") as f:
            json.dump(output_data, f, ensure_ascii=False, indent=2)
        
        # CSV détaillé
        if results:
            csv_data = []
            for result in results:
                csv_data.append({
                    "tweet_url": result.get("tweet_url", ""),
                    "title": result.get("title", ""),
                    "author": result.get("author", ""),
                    "timestamp": result.get("timestamp", ""),
                    "likes": result.get("metrics", {}).get("likes", 0),
                    "retweets": result.get("metrics", {}).get("retweets", 0),
                    "scraped_at": result.get("scraped_at", "")
                })
            
            df = pd.DataFrame(csv_data)
            df.to_csv("hespress_TWITTER_septembre_2025_FINAL.csv", index=False, encoding="utf-8-sig")
        
        # Fichier texte simple avec URLs Twitter
        with open("hespress_TWITTER_urls_septembre_2025_FINAL.txt", "w", encoding="utf-8") as f:
            for result in results:
                f.write(result.get("tweet_url", "") + "\n")
        
        logging.info("💾 FICHIERS TWITTER SAUVEGARDÉS:")
        logging.info("   - hespress_TWITTER_septembre_2025_FINAL.json")
        logging.info("   - hespress_TWITTER_septembre_2025_FINAL.csv")
        logging.info("   - hespress_TWITTER_urls_septembre_2025_FINAL.txt")
        logging.info("   - hespress_twitter_urls_septembre_2025_COMPLET.txt (tous les liens Twitter)")
        
    except Exception as e:
        logging.error(f"❌ Erreur sauvegarde Twitter: {e}")

# --------------------------
# VÉRIFICATION TWITTER SEULEMENT
# --------------------------
def check_twitter_results():
    """Vérification des résultats Twitter seulement"""
    try:
        files = {
            "Fichier JSON Twitter": "hespress_TWITTER_septembre_2025_FINAL.json",
            "Fichier CSV Twitter": "hespress_TWITTER_septembre_2025_FINAL.csv", 
            "URLs Twitter septembre 2025": "hespress_TWITTER_urls_septembre_2025_FINAL.txt",
            "Tous les liens Twitter": "hespress_twitter_urls_septembre_2025_COMPLET.txt"
        }
        
        for nom, fichier in files.items():
            if os.path.exists(fichier):
                taille = os.path.getsize(fichier)
                print(f"✅ {nom}: {fichier} ({taille} octets)")
                
                if fichier.endswith('.json') and taille > 0:
                    with open(fichier, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                        if "tweets" in data:
                            print(f"   📊 {len(data['tweets'])} tweets septembre 2025")
                            print(f"   🎯 Plateforme: {data.get('metadata', {}).get('platform', 'Twitter/X')}")
                        else:
                            print(f"   📊 {len(data)} tweets septembre 2025")
                elif fichier.endswith('.txt'):
                    with open(fichier, 'r', encoding='utf-8') as f:
                        lignes = f.readlines()
                        print(f"   🔗 {len(lignes)} URLs Twitter")
                        # Afficher quelques exemples
                        if lignes:
                            print(f"   📝 Exemples: {lignes[0].strip()}")
                            if len(lignes) > 1:
                                print(f"              {lignes[1].strip()}")
            else:
                print(f"❌ {nom}: {fichier} - NON TROUVÉ")
                
    except Exception as e:
        print(f"❌ Erreur vérification: {e}")

# --------------------------
# EXÉCUTION PRINCIPALE
# --------------------------
if __name__ == "__main__":
    print("=" * 70)
    print("🐦 SCRAPER TWITTER/X UNIQUEMENT - HESPRESS SEPTEMBRE 2025")
    print("=" * 70)
    print("🎯 CE SCRAPER VA:")
    print("   - Rechercher UNIQUEMENT sur Twitter/X")
    print("   - IGNORER YouTube, Instagram, Facebook, TikTok, etc.")
    print("   - Filtrer AUTOMATIQUEMENT septembre 2025")
    print("   - Utiliser 'site:x.com' dans toutes les requêtes")
    print("   - Sauvegarder uniquement les tweets valides")
    print("=" * 70)
    
    print("Options:")
    print("1. 🚀 Lancer le scraping TWITTER SEULEMENT")
    print("2. 🔍 Vérifier les résultats Twitter")
    
    choix = input("\nVotre choix (1 ou 2): ").strip()
    
    if choix == "1":
        print("🚀 LANCEMENT SCRAPING TWITTER UNIQUEMENT...")
        print("⏰ Cette opération peut prendre 10-20 minutes.")
        print("🎯 Objectif: Tweets Hespress de septembre 2025 UNIQUEMENT")
        confirmation = input("Confirmez-vous? (o/n): ").strip().lower()
        
        if confirmation == 'o':
            scrape_hespress_twitter_september_2025()
        else:
            print("❌ Opération annulée.")
    elif choix == "2":
        print("🔍 Vérification des fichiers Twitter...")
        check_twitter_results()
    else:
        print("❌ Choix invalide. Veuillez choisir 1 ou 2.")

🐦 SCRAPER TWITTER/X UNIQUEMENT - HESPRESS SEPTEMBRE 2025
🎯 CE SCRAPER VA:
   - Rechercher UNIQUEMENT sur Twitter/X
   - IGNORER YouTube, Instagram, Facebook, TikTok, etc.
   - Filtrer AUTOMATIQUEMENT septembre 2025
   - Utiliser 'site:x.com' dans toutes les requêtes
   - Sauvegarder uniquement les tweets valides
Options:
1. 🚀 Lancer le scraping TWITTER SEULEMENT
2. 🔍 Vérifier les résultats Twitter



Votre choix (1 ou 2):  1


🚀 LANCEMENT SCRAPING TWITTER UNIQUEMENT...
⏰ Cette opération peut prendre 10-20 minutes.
🎯 Objectif: Tweets Hespress de septembre 2025 UNIQUEMENT


Confirmez-vous? (o/n):  O


2025-10-25 19:14:41,121 - INFO - ====== WebDriver manager ======
2025-10-25 19:14:46,735 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-25 19:14:46,905 - INFO - Get LATEST chromedriver version for google-chrome
2025-10-25 19:14:47,044 - INFO - Driver [C:\Users\dell\.wdm\drivers\chromedriver\win64\141.0.7390.122\chromedriver-win32/chromedriver.exe] found in cache
2025-10-25 19:14:48,329 - INFO - 🎯 Recherche Twitter 1/12: site:x.com hespress video september 2025
2025-10-25 19:14:48,333 - INFO - 🔍 Recherche Google Twitter ONLY: site:x.com hespress video september 2025
2025-10-25 19:14:53,136 - INFO - 📜 Scroll pour charger plus de résultats...
2025-10-25 19:15:05,377 - INFO - ✅ Lien Twitter trouvé: https://x.com/hespress/status/1965520589206229426
2025-10-25 19:15:05,432 - INFO - ✅ Lien Twitter trouvé: https://x.com/hespress/status/1973170561812668582
2025-10-25 19:15:05,464 - INFO - ✅ Lien Twitter trouvé: https://x.com/hespress/status/1970801323374510109
2025-10-25 19:

In [4]:
import pandas as pd
import os
from urllib.parse import urlparse

def extract_and_analyze_tweet_urls(csv_file_path):
    """
    Extrait les URLs de tweets et fournit une analyse détaillée
    """
    try:
        # Lire le CSV
        df = pd.read_csv(csv_file_path)
        
        print(f"📊 ANALYSE DU FICHIER:")
        print(f"   - Nombre de lignes: {len(df)}")
        print(f"   - Colonnes: {list(df.columns)}")
        
        # Vérifier la colonne tweet_url
        if 'tweet_url' not in df.columns:
            print("❌ Colonne 'tweet_url' non trouvée")
            return []
        
        # Extraire les URLs
        urls = df['tweet_url'].dropna().tolist()
        clean_urls = [url.strip() for url in urls if isinstance(url, str) and url.strip()]
        
        print(f"✅ EXTRACTION RÉUSSIE:")
        print(f"   - URLs trouvées: {len(clean_urls)}")
        print(f"   - URLs uniques: {len(set(clean_urls))}")
        
        # Analyser les domaines
        domains = []
        for url in clean_urls:
            try:
                domain = urlparse(url).netloc
                domains.append(domain)
            except:
                pass
        
        domain_count = pd.Series(domains).value_counts()
        print(f"   - Domaines: {dict(domain_count)}")
        
        return clean_urls
        
    except Exception as e:
        print(f"❌ Erreur: {e}")
        return []

# Utilisation
csv_file = "hespress_TWITTER_septembre_2025_95_yrl.csv"

# Extraire les URLs
urls_list = extract_and_analyze_tweet_urls(csv_file)

# Afficher la liste complète
if urls_list:
    print(f"\n🎯 LISTE COMPLÈTE ({len(urls_list)} URLs):")
    print("[" + ",\n ".join([f'"{url}"' for url in urls_list]) + "]")
    
    # Copier-coller facile
    print(f"\n📋 POUR COPIER-COLLER:")
    print(f"urls = {urls_list}")

📊 ANALYSE DU FICHIER:
   - Nombre de lignes: 94
   - Colonnes: ['tweet_url', 'title', 'author', 'timestamp', 'likes', 'retweets', 'scraped_at']
✅ EXTRACTION RÉUSSIE:
   - URLs trouvées: 94
   - URLs uniques: 94
   - Domaines: {'x.com': 92, 'twitter.com': 2}

🎯 LISTE COMPLÈTE (94 URLs):
["https://x.com/hespress/status/1963950250860613657",
 "https://x.com/hespress/status/1963534735843418568",
 "https://twitter.com/hespress/status/1966924845989720468/video/1",
 "https://x.com/hespress/status/1965380917775478899",
 "https://x.com/hespress/status/1968457441248723444",
 "https://x.com/hespress/status/1971567935027351565",
 "https://x.com/hespress/status/1970833278635856234",
 "https://x.com/hespress/status/1969338342660517893",
 "https://x.com/hespress/status/1962621486625353965",
 "https://x.com/hespress/status/1973070339849740758",
 "https://x.com/hespress/status/1968687964310954183",
 "https://x.com/hespress/status/1964418328354762920",
 "https://x.com/hespress/status/1970968340404502675